### **Conventional Workflow**

#### Import Libraries

In [9]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt

#### Load the data

In [10]:
from ucimlrepo import fetch_ucirepo

breast_cancer = fetch_ucirepo(id=17)

#### Explore the Data

In [11]:
X = breast_cancer.data.features
y = breast_cancer.data.targets.squeeze()
X.shape, y.shape, X.info(), y.info(), y.value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  concavity2         

((569, 30),
 (569,),
 None,
 None,
 Diagnosis
 B    357
 M    212
 Name: count, dtype: int64)

In [12]:
features = breast_cancer.data.features.columns.tolist()

#### Check for duplicates

In [13]:
X.duplicated().sum()

np.int64(0)

In [14]:
y = y.replace({"B": 0, "M": 1})
y = y.astype(int)

#### Split the dataset

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((455, 30), (455,), (114, 30), (114,))

**Preprocessing**: The dataset contains no missing values, and all predictor variables are numeric, so imputation and one-hot encoding are not required. Standardization is applied within the pipelines for Logistic Regression and SVC. Tree-based models (Random Forest, Gradient Boosting, and XGBoost) are evaluated without feature scaling.

####  Define Candidate Model Pipelines

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb

GB_pipeline = Pipeline([
    ("clf", GradientBoostingClassifier(random_state=42))
])

XGB_pipeline = Pipeline([
    ("clf", xgb.XGBClassifier(random_state=42))
])

LR_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression())
])

RF_pipeline = Pipeline([
    ('clf', RandomForestClassifier(random_state=42))
])

SVC_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ('clf', SVC(kernel='rbf', C=1.0, gamma='scale'))
])

### Standardized Cross Validation Evaluation Function

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
import time
import numpy as np

skfolds = StratifiedKFold(n_splits=3, random_state=42, shuffle=True)


In [18]:

def evaluate_model(model_name, model, x, y, cv):

    y_true_all = []
    y_pred_all = []
    y_score_all = []

    # NEW: Store metrics for each CV fold
    fold_results = []

    start_time = time.perf_counter()
    fit_count = 0

    for train_index, val_index in cv.split(X=x, y=y):

        clone_clf = clone(model)

        X_train_folds = x.iloc[train_index]
        y_train_folds = y.iloc[train_index]

        X_val_folds = x.iloc[val_index]
        y_val_folds = y.iloc[val_index]

        # ONE fit for this fold
        clone_clf.fit(X_train_folds, y_train_folds)
        fit_count += 1

        # Class predictions
        y_pred = clone_clf.predict(X_val_folds)

        # Continuous scores for AUC
        if hasattr(clone_clf, "predict_proba"):
            y_score = clone_clf.predict_proba(
                X_val_folds
            )[:, 1]

        elif hasattr(clone_clf, "decision_function"):
            y_score = clone_clf.decision_function(
                X_val_folds
            )

        else:
            raise ValueError(
                f"{model_name} cannot produce scores for AUC"
            )

        # Save this fold's predictions
        y_true_all.extend(y_val_folds)
        y_pred_all.extend(y_pred)
        y_score_all.extend(y_score)

        # NEW: Calculate metrics for each fold
        fold_results.append({
            "Accuracy": accuracy_score(y_val_folds, y_pred),
            "Precision": precision_score(
                y_val_folds, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_val_folds, y_pred, zero_division=0
            ),
            "F1": f1_score(
                y_val_folds, y_pred, zero_division=0
            ),
            "AUC": roc_auc_score(y_val_folds, y_score)
        })

    runtime = time.perf_counter() - start_time

    # Convert to arrays
    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_score_all = np.array(y_score_all)

    # Original pooled metrics
    accuracy = accuracy_score(y_true_all, y_pred_all)
    precision = precision_score(y_true_all, y_pred_all)
    recall = recall_score(y_true_all, y_pred_all)
    f1 = f1_score(y_true_all, y_pred_all)
    auc = roc_auc_score(y_true_all, y_score_all)

    # NEW: Calculate mean and SD across folds
    folds_df = pd.DataFrame(fold_results)

    metrics = [
        "Accuracy", "Precision", "Recall", "F1", "AUC"
    ]

    means = folds_df[metrics].mean()
    stds = folds_df[metrics].std(ddof=1)

    cm = confusion_matrix(
        y_true=y_true_all,
        y_pred=y_pred_all
    )

    return {
        "Model": model_name,

        # Original pooled metrics
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "AUC": auc,

        # NEW: Mean and standard deviation
        "Accuracy_mean": means["Accuracy"],
        "Accuracy_std": stds["Accuracy"],

        "Precision_mean": means["Precision"],
        "Precision_std": stds["Precision"],

        "Recall_mean": means["Recall"],
        "Recall_std": stds["Recall"],

        "F1_mean": means["F1"],
        "F1_std": stds["F1"],

        "AUC_mean": means["AUC"],
        "AUC_std": stds["AUC"],

        "Runtime_seconds": runtime,
        "Model_evaluations": 1,
        "CV_fits": fit_count,
        "Confusion_matrix": cm,
        "Fold_results": fold_results
    }


### **AI Agent Workflow**


In [ ]:
import os
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face token: ")

In [ ]:
from smolagents import (
    ToolCallingAgent, InferenceClientModel,
)  

In [ ]:
model = InferenceClientModel(
    model_id="meta-llama/Llama-3.1-8B-Instruct",
    provider="deepinfra",
    token=HF_TOKEN,
    temperature=0
)

In [185]:
ML_MODELS = {
    "LR": LR_pipeline,
    "SVC": SVC_pipeline,
    "RF": RF_pipeline,
    "GB": GB_pipeline,
    "XGB": XGB_pipeline
}

In [186]:
ALLOWED_MODELS = list(ML_MODELS.keys())

In [187]:
action_log = []
all_experiment_logs = []
all_action_logs = []
agent_run_summary = []
experiment_log = []
experiment_context = {}

In [188]:

MAX_EVALUATIONS = 7

agent_state = {
    "evaluations": 0,
    "selected_features": list(X_train.columns),
    "level3_phase": "inspect"
}

In [189]:
def save_run_logs(level, run_id):

    for record in experiment_log:
        saved = record.copy()
        saved["level"] = level
        saved["run_id"] = run_id
        all_experiment_logs.append(saved)

    for record in action_log:
        saved = record.copy()
        saved["level"] = level
        saved["run_id"] = run_id
        all_action_logs.append(saved)

In [190]:
def reset_agent_state():
    agent_state["evaluations"] = 0
    agent_state["selected_features"] = list(X_train.columns)
    agent_state["level3_phase"] = "inspect"
    
    experiment_log.clear()
    action_log.clear()

In [191]:
import json
import time
import joblib
import numpy as np
import pandas as pd

from copy import deepcopy
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
from smolagents import tool, ToolCallingAgent

Recored Token Usage, API cost and Runtime for LLM

In [192]:
import time


INPUT_PRICE_PER_MILLION = 0.02
OUTPUT_PRICE_PER_MILLION = 0.05


def get_agent_usage(agent):
    """
    Extract token usage from the most recent smolagents run.
    Call after agent.run(..., reset=True).
    """

    steps = getattr(
        getattr(agent, "memory", None),
        "steps",
        []
    )

    input_tokens = 0
    output_tokens = 0
    found_input = False
    found_output = False

    for step in steps:
        usage = getattr(step, "token_usage", None)

        if usage is None:
            continue

        def read_field(obj, names):
            for name in names:
                value = (
                    obj.get(name)
                    if isinstance(obj, dict)
                    else getattr(obj, name, None)
                )
                if value is not None:
                    return int(value)
            return None

        inp = read_field(
            usage,
            ["input_tokens", "prompt_tokens"]
        )

        out = read_field(
            usage,
            ["output_tokens", "completion_tokens"]
        )

        if inp is not None:
            input_tokens += inp
            found_input = True

        if out is not None:
            output_tokens += out
            found_output = True

    input_tokens = input_tokens if found_input else None
    output_tokens = output_tokens if found_output else None

    total_tokens = (
        input_tokens + output_tokens
        if input_tokens is not None
        and output_tokens is not None
        else None
    )

    api_cost = (
        (
            input_tokens * INPUT_PRICE_PER_MILLION
            + output_tokens * OUTPUT_PRICE_PER_MILLION
        ) / 1_000_000
        if total_tokens is not None
        and INPUT_PRICE_PER_MILLION is not None
        and OUTPUT_PRICE_PER_MILLION is not None
        else None
    )

    return {
        "Input_Tokens": input_tokens,
        "Output_Tokens": output_tokens,
        "Total_Tokens": total_tokens,
        "API_Cost_USD": api_cost
    }

#### Level 1 - Fixed Prediction

experiment context and state

In [193]:
import time
import pandas as pd
from smolagents import tool, ToolCallingAgent

# Training data available to Level 1
experiment_context = {
    "X": X_train,
    "y": y_train
}

# Level 1 state
agent_state = {
    "evaluations": 0,
    "completed": False
}

action_log = []

evaluate classifier for level 1

In [194]:
@tool
def evaluate_fixed_classifier() -> str:
    """
    Evaluate fixed Logistic Regression using all training features
    and the predefined cross-validation procedure.
    """

    # Allow only one evaluation
    if agent_state["evaluations"] >= 1:
        return "ERROR: Fixed classifier already evaluated."

    # Use current experiment's training data
    X = experiment_context["X"]
    y = experiment_context["y"]

    # Evaluate fixed Logistic Regression
    result = evaluate_model(
        model_name="LR",
        model=LR_pipeline,
        x=X,
        y=y,
        cv=skfolds
    )

    # Count successful evaluation
    agent_state["evaluations"] += 1

    # Record evaluation results
    experiment_log.append({
        "step": len(experiment_log) + 1,
        "level": 1,
        "model": "LR",
        "features": X.columns.tolist(),
        "n_features": X.shape[1],

        "accuracy": result["Accuracy"],
        "precision": result["Precision"],
        "recall": result["Recall"],

        "f1": result["F1"],
        "f1_mean": result["F1_mean"],
        "f1_std": result["F1_std"],

        "auc": result["AUC"],

        "runtime": result["Runtime_seconds"],
        "model_evaluations": result["Model_evaluations"],
        "cv_fits": result["CV_fits"]
    })

    return str(result)

In [195]:
l1_agent = ToolCallingAgent(
    tools=[evaluate_fixed_classifier],
    model=model,
    max_steps=10
)

In [196]:
LEVEL1_PROMPT = """Your goal is to make use of the tool given to you to complete breast-cancer
classification task.

You may only use the classifier given to you

Use only the available tool and its cross-validation results.
The held-out test set is unavailable.

You have a limited evaluation budget.

When finished, report the classifier that was used and the validation
evidence supporting the decision."""

In [197]:
reset_agent_state()

start_agent = time.perf_counter()

try:
    level1_result = l1_agent.run(
        LEVEL1_PROMPT,
        reset=True
    )

finally:
    level1_agent_runtime = (
        time.perf_counter() - start_agent
    )

    level1_usage = get_agent_usage(l1_agent)


# Create Level 1 usage summary
level1_usage_summary = pd.DataFrame([{
    "Level": 1,
    "Run_ID": 1,
    "Agent_Runtime": level1_agent_runtime,
    **level1_usage
}])
 
with pd.option_context("display.max_colwidth", None, "display.width", None):
    display(level1_usage_summary.round(6))

# Save existing experiment logs
save_run_logs(
    level=1,
    run_id=1
)

# Save token usage and API cost separately
level1_usage_summary.to_csv(
    "level1_run_1_usage.csv",
    index=False
)

level1_experiment_df = pd.DataFrame(experiment_log)

# Add experiment identifiers
level1_experiment_df.insert(0, "Level", 1)
level1_experiment_df.insert(1, "Run_ID", 1)

# Display all experiment results
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None
):
    display(level1_experiment_df)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Your goal is to make use of the tool given to you to complete breast-cancer                                     │
│ classification task.                                                                                            │
│                                                                                                                 │
│ You may only use the classifier given to you                                                                    │
│                                                                                                                 │
│ Use only the available tool and its cross-validation results.                                                   │
│ The held-out test set is unavailable.                                                                           │
│                                                                                                                 │
│ You have a limited evaluation budget.                                                                           │
│                                                                                                                 │
│ When finished, report the classifier that was used and the validation                                           │
│ evidence supporting the decision.                                                                               │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.1-8B-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'evaluate_fixed_classifier' with arguments: {}                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {'Model': 'LR', 'Accuracy': 0.9736263736263736, 'Precision': 0.9702380952380952, 'Recall': 
0.9588235294117647, 'F1': 0.9644970414201184, 'AUC': 0.9951909184726522, 'Accuracy_mean': 
np.float64(0.9736261182758219), 'Accuracy_std': np.float64(0.013158279448108674), 'Precision_mean': 
np.float64(0.9705513784461153), 'Precision_std': np.float64(0.026869400554375415), 'Recall_mean': 
np.float64(0.9588554720133667), 'Recall_std': np.float64(0.009953009655341601), 'F1_mean': 
np.float64(0.9645989974937343), 'F1_std': np.float64(0.01738933469470917), 'AUC_mean': 
np.float64(0.995510706591039), 'AUC_std': np.float64(0.00470927675276659), 'Runtime_seconds': 0.17111220001243055, 
'Model_evaluations': 1, 'CV_fits': 3, 'Confusion_matrix': array(||280,   5],
       |  7, 163]]), 'Fold_results': |{'Accuracy': 0.9605263157894737, 'Precision': 0.9473684210526315, 'Recall': 
0.9473684210526315, 'F1': 0.9473684210526315, 'AUC': 0.997045244690674}, {'Accuracy': 0.9868421052631579, 
'Precision': 1.0, 'Recall': 0.9649122807017544, 'F1': 0.9821428571428571, 'AUC': 0.9992613111726685}, {'Accuracy': 
0.9735099337748344, 'Precision': 0.9642857142857143, 'Recall': 0.9642857142857143, 'F1': 0.9642857142857143, 'AUC':
0.9902255639097745}]}

[Step 1: Duration 1.46 seconds| Input tokens: 1,214 | Output tokens: 11]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (2 outputs left out)

Final answer: The classifier used is Logistic Regression (LR) with an accuracy of 0.9736, precision of 0.9702, 
recall of 0.9588, F1 score of 0.9645, and AUC of 0.9952. The cross-validation results show consistent performance 
across the three folds, with accuracy ranging from 0.9605 to 0.9868, precision from 0.9474 to 1.0, recall from 
0.9474 to 0.9649, F1 score from 0.9474 to 0.9821, and AUC from 0.9970 to 0.9993.

[Step 2: Duration 5.42 seconds| Input tokens: 2,997 | Output tokens: 178]

,Level,Run_ID,Agent_Runtime,Input_Tokens,Output_Tokens,Total_Tokens,API_Cost_USD
0,1,1,6.898551,2997,178,3175,0.000069


,Level,Run_ID,step,level,model,features,n_features,accuracy,precision,recall,f1,f1_mean,f1_std,auc,runtime,model_evaluations,cv_fits
0,1,1,1,1,LR,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, fractal_dimension1, radius2, texture2, perimeter2, area2, smoothness2, compactness2, concavity2, concave_points2, symmetry2, fractal_dimension2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]",30,0.973626,0.970238,0.958824,0.964497,0.964599,0.017389,0.995191,0.171112,1,3


In [ ]:
final_level1_model = clone(LR_pipeline)
final_level1_model.fit(X_train, y_train)

joblib.dump(
    final_level1_model,
    "level1_final_model.joblib"
)2

#### Level 2 - Feature Selection

In [198]:
from sklearn.feature_selection import SelectKBest, f_classif
# One context per outer fold
level2_context = {}

level2_fold_results = []
level2_action_log = []

Inspect training data

In [199]:
@tool
def inspect_data() -> str:
    """
    Return summary information about the current
    permitted training data.
    """

    X = level2_context["X"]
    y = level2_context["y"]

    summary = {
        "rows": X.shape[0],
        "features": X.shape[1],
        "missing_values": int(X.isnull().sum().sum()),
        "class_distribution": y.value_counts().to_dict(),
        "feature_names": X.columns.tolist()
    }

    level2_action_log.append({
        "fold": level2_context["fold"],
        "action": "inspect_data"
    })

    return str(summary)

Path A - Select features from pre-existing knowledge

In [200]:
@tool
def level2_select_features(features: list[str]) -> str:
    """
    Select feature names directly.

    Args:
        features: List of valid feature names returned by inspect_data.
    """

    if level2_context["method"] is not None:
        return "ERROR: Feature-selection path is already locked."

    if level2_context.get("anova_inspected", False):
        return (
            "ERROR: ANOVA was already inspected. "
            "Knowledge-only selection is unavailable."
        )

    X = level2_context["X"]

    if not features or len(features) != len(set(features)):
        return "ERROR: Provide a nonempty list without duplicates."

    invalid = [f for f in features if f not in X.columns]

    if invalid:
        return f"ERROR: Invalid features: {invalid}"

    level2_context["method"] = "manual"
    level2_context["features"] = features.copy()
    level2_context["k"] = len(features)

    level2_action_log.append({
        "fold": level2_context["fold"],
        "action": "select_features",
        "method": "manual",
        "features": features.copy(),
        "n_features": len(features)
    })

    print(f"Selected {len(features)} features.")

    return f"Path A locked: {features}"

optional ANOVA analysis tool

In [201]:
@tool
def level2_analyze_anova() -> str:
    """Return ANOVA F-scores and p-values for the current training fold."""
    if level2_context["method"] is not None:
        return "ERROR: Feature-selection decision is already locked."

    X = level2_context["X"]
    y = level2_context["y"]

    scores, p_values = f_classif(X, y)

    ranking = pd.DataFrame({
        "feature": X.columns,
        "f_score": scores,
        "p_value": p_values
    }).sort_values("f_score", ascending=False)

    level2_context["anova_inspected"] = True

    level2_action_log.append({
        "fold": level2_context["fold"],
        "action": "analyze_anova"
    })

    return ranking.to_json(orient="records")

Path B - ANOVA + agent selected K

In [202]:
@tool
def level2_select_kbest(k: int) -> str:
    """
    Choose the number of features retained by ANOVA SelectKBest.

    Args:
        k: Number of features to retain.
    """
    if level2_context["method"] is not None:
        return "ERROR: Feature-selection path is already locked."

    if not 1 <= k <= level2_context["X"].shape[1]:
        return "ERROR: Invalid k."

    level2_context["method"] = "anova_kbest"
    level2_context["features"] = None
    level2_context["k"] = k

    level2_action_log.append({
    "fold": level2_context["fold"],
    "action": "select_kbest",
    "method": "anova_kbest",
    "k": k
})

    return f"Path B locked: ANOVA SelectKBest with k={k}."

Path B - ANOVA + Automated K

In [203]:
@tool
def level2_search_best_k() -> str:
    """
    Choose automatic feature-count selection using
    ANOVA SelectKBest and GridSearchCV.
    """
    if level2_context["method"] is not None:
        return "ERROR: Feature-selection path is already locked."

    level2_context["method"] = "anova_gridsearch"
    level2_context["features"] = None
    level2_context["k"] = None

    level2_action_log.append({
    "fold": level2_context["fold"],
    "action": "search_best_k",
    "method": "anova_gridsearch"
})

    return (
        "Path C locked. GridSearchCV will automatically "
        "determine k using inner cross-validation F1."
    )

In [204]:
l2_agent = ToolCallingAgent(
    tools=[
        inspect_data,
        level2_analyze_anova,
        level2_select_features,
        level2_select_kbest,
        level2_search_best_k

    ],
    model=model,
    max_steps=10
)

Level 2 prompt

In [205]:
LEVEL2_PROMPT = """
You are performing Level 2: Agent-Directed Feature Selection.

GOAL:
Choose a feature-selection strategy for breast-cancer classification.

The classifier is fixed as Logistic Regression.
You cannot change its model or hyperparameters.

WORKFLOW:
1. Inspect the available training features.
2. Optionally inspect ANOVA rankings and scores.
3. Choose exactly ONE feature-selection path:
   A. Select feature names directly using your knowledge.
   B. Choose k for ANOVA SelectKBest.
   C. Use GridSearchCV to automatically determine k
      for ANOVA SelectKBest.
4. Once a valid path is selected, it is locked.
5. Finish by explaining your chosen method.

RULES:
- You decide whether ANOVA or automated search is necessary.
- If choosing Path B, use available ANOVA evidence to inform k.
- Consider the computational cost of each method.
- Use only valid feature names.
- Pass manual features as a list of separate strings.
- Never change paths after selection.
- Never access the held-out test set.
- Do not invent evaluation results.

FINAL ANSWER:
Report the selected path, features or k, and why you chose it.
"""

Fit selected feature-selection strategy

In [206]:
from sklearn.model_selection import GridSearchCV

def fit_level2_configuration(X, y):

    method = level2_context["method"]

    if method is None:
        raise ValueError("Agent did not select a path.")

    if method == "manual":

        features = level2_context["features"]

        fitted_model = clone(LR_pipeline)
        fitted_model.fit(X[features], y)

        return fitted_model, features, len(features), 1

    elif method == "anova_kbest":

        k = level2_context["k"]

        fitted_model = Pipeline([
            ("selector", SelectKBest(f_classif, k=k)),
            ("classifier", clone(LR_pipeline))
        ])

        fitted_model.fit(X, y)

        selector = fitted_model.named_steps["selector"]

        features = X.columns[
            selector.get_support()
        ].tolist()

        return fitted_model, features, k, 1

    elif method == "anova_gridsearch":

        candidate = Pipeline([
            ("selector", SelectKBest(f_classif)),
            ("classifier", clone(LR_pipeline))
        ])

        search = GridSearchCV(
            estimator=candidate,
            param_grid={
                "selector__k": list(range(1, X.shape[1] + 1))
            },
            scoring="f1",
            cv=skfolds,
            refit=True,
            n_jobs=-1
        )

        search.fit(X, y)

        fitted_model = search.best_estimator_

        k = search.best_params_["selector__k"]

        selector = fitted_model.named_steps["selector"]

        features = X.columns[
            selector.get_support()
        ].tolist()

        total_fits = (
            len(search.cv_results_["params"])
            * search.n_splits_
            + 1
        )

        return fitted_model, features, k, total_fits

    else:
        raise ValueError(f"Unknown method: {method}")

running outer loop

In [207]:

level2_fold_results = []
level2_action_log = []

start_total = time.perf_counter()

for fold, (train_idx, val_idx) in enumerate(
    skfolds.split(X_train, y_train), start=1
):

    X_outer_train = X_train.iloc[train_idx]
    y_outer_train = y_train.iloc[train_idx]

    X_outer_val = X_train.iloc[val_idx]
    y_outer_val = y_train.iloc[val_idx]

    # Reset Level 2 state
    level2_context.clear()

    level2_context.update({
        "X": X_outer_train,
        "y": y_outer_train,
        "fold": fold,
        "method": None,
        "features": None,
        "k": None,
        "anova_inspected": False
    })

    start_agent = time.perf_counter()

    # Default usage in case the agent fails
    level2_usage = {
        "Input_Tokens": None,
        "Output_Tokens": None,
        "Total_Tokens": None,
        "API_Cost_USD": None
    }

    try:
        # Agent chooses feature-selection strategy
        agent_result = l2_agent.run(
            LEVEL2_PROMPT,
            reset=True
        )

        agent_runtime = time.perf_counter() - start_agent

        # Collect usage for this fold
        level2_usage = get_agent_usage(l2_agent)

        # Ensure valid feature path
        if level2_context["method"] is None:
            raise ValueError(
                "Agent did not select a valid feature path."
            )

        # Fit selected configuration
        start_ml = time.perf_counter()

        fitted_model, selected_features, selected_k, fits = (
            fit_level2_configuration(
                X_outer_train,
                y_outer_train
            )
        )

        # Prepare validation features
        if level2_context["method"] == "manual":
            X_eval = X_outer_val[selected_features]
        else:
            X_eval = X_outer_val

        # Predict on outer validation fold
        y_pred = fitted_model.predict(X_eval)

        if hasattr(fitted_model, "predict_proba"):
            pos_idx = list(fitted_model.classes_).index(1)

            y_score = fitted_model.predict_proba(
                X_eval
            )[:, pos_idx]

        elif hasattr(fitted_model, "decision_function"):
            y_score = fitted_model.decision_function(X_eval)

            if fitted_model.classes_[1] != 1:
                y_score = -y_score

        else:
            raise ValueError(
                "Classifier cannot produce AUC scores."
            )

        ml_runtime = time.perf_counter() - start_ml

        level2_fold_results.append({
            "Fold": fold,
            "Status": "completed",
            "Method": level2_context["method"],
            "ANOVA_Inspected": level2_context["anova_inspected"],
            "Selected_k": selected_k,
            "Selected_Features": selected_features,

            "Accuracy": accuracy_score(y_outer_val, y_pred),
            "Precision": precision_score(
                y_outer_val, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_outer_val, y_pred, zero_division=0
            ),
            "F1": f1_score(
                y_outer_val, y_pred, zero_division=0
            ),
            "AUC": roc_auc_score(y_outer_val, y_score),

            "Agent_Runtime": agent_runtime,
            "ML_Runtime": ml_runtime,
            "Total_Model_Fits": fits,

            "Model_Evaluations": (
                X_outer_train.shape[1]
                if level2_context["method"] == "anova_gridsearch"
                else 1
            ),

            # Token usage and API cost
            **{
                key: level2_usage.get(key)
                for key in [
                    "Input_Tokens",
                    "Output_Tokens",
                    "Total_Tokens",
                    "API_Cost_USD"
                ]
            }
        })

    except Exception as e:

        # Capture available usage even if a fold fails
        try:
            level2_usage = get_agent_usage(l2_agent)
        except Exception:
            pass

        level2_fold_results.append({
            "Fold": fold,
            "Status": "failed",
            "Method": level2_context.get("method"),
            "Error": str(e),
            "Agent_Runtime": time.perf_counter() - start_agent,
            **{
                key: level2_usage.get(key)
                for key in [
                    "Input_Tokens",
                    "Output_Tokens",
                    "Total_Tokens",
                    "API_Cost_USD"
                ]
            }
        })

total_runtime = time.perf_counter() - start_total

level2_results_df = pd.DataFrame(level2_fold_results)

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None
):
    display(level2_results_df.round(4))


level2_usage_summary = pd.DataFrame([{
    "Level": 2,
    "Agent_Runtime": level2_results_df["Agent_Runtime"].sum(),
    "Input_Tokens": level2_results_df["Input_Tokens"].sum(min_count=1),
    "Output_Tokens": level2_results_df["Output_Tokens"].sum(min_count=1),
    "Total_Tokens": level2_results_df["Total_Tokens"].sum(min_count=1),
    "API_Cost_USD": level2_results_df["API_Cost_USD"].sum(min_count=1)
}])

display(level2_usage_summary.round(6))

# Save existing experiment logs
save_run_logs(
    level=2,
    run_id=1
)

# Save detailed fold results
level2_results_df.to_csv(
    "level2_run_1_results.csv",
    index=False
)

# Save total token usage and cost
level2_usage_summary.to_csv(
    "level2_run_1_usage.csv",
    index=False
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are performing Level 2: Agent-Directed Feature Selection.                                                   │
│                                                                                                                 │
│ GOAL:                                                                                                           │
│ Choose a feature-selection strategy for breast-cancer classification.                                           │
│                                                                                                                 │
│ The classifier is fixed as Logistic Regression.                                                                 │
│ You cannot change its model or hyperparameters.                                                                 │
│                                                                                                                 │
│ WORKFLOW:                                                                                                       │
│ 1. Inspect the available training features.                                                                     │
│ 2. Optionally inspect ANOVA rankings and scores.                                                                │
│ 3. Choose exactly ONE feature-selection path:                                                                   │
│    A. Select feature names directly using your knowledge.                                                       │
│    B. Choose k for ANOVA SelectKBest.                                                                           │
│    C. Use GridSearchCV to automatically determine k                                                             │
│       for ANOVA SelectKBest.                                                                                    │
│ 4. Once a valid path is selected, it is locked.                                                                 │
│ 5. Finish by explaining your chosen method.                                                                     │
│                                                                                                                 │
│ RULES:                                                                                                          │
│ - You decide whether ANOVA or automated search is necessary.                                                    │
│ - If choosing Path B, use available ANOVA evidence to inform k.                                                 │
│ - Consider the computational cost of each method.                                                               │
│ - Use only valid feature names.                                                                                 │
│ - Pass manual features as a list of separate strings.                                                           │
│ - Never change paths after selection.                                                                           │
│ - Never access the held-out test set.                                                                           │
│ - Do not invent evaluation results.                                                                             │
│                                                                                                                 │
│ FINAL ANSWER:                                                                                                   │
│ Report the selected path, features or k, and why you chose it.                                                  │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.1-8B-Instru

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'inspect_data' with arguments: {}                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {'rows': 303, 'features': 30, 'missing_values': 0, 'class_distribution': {0: 190, 1: 113}, 
'feature_names': |'radius1', 'texture1', 'perimeter1', 'area1', 'smoothness1', 'compactness1', 'concavity1', 
'concave_points1', 'symmetry1', 'fractal_dimension1', 'radius2', 'texture2', 'perimeter2', 'area2', 'smoothness2', 
'compactness2', 'concavity2', 'concave_points2', 'symmetry2', 'fractal_dimension2', 'radius3', 'texture3', 
'perimeter3', 'area3', 'smoothness3', 'compactness3', 'concavity3', 'concave_points3', 'symmetry3', 
'fractal_dimension3']}

[Step 1: Duration 1.24 seconds| Input tokens: 1,937 | Output tokens: 29]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (46 outputs left out)

Final answer: Path B: ANOVA SelectKBest with k=10. I chose this method because it allows for a balance between 
feature selection and model performance, and it is a well-established method in the field of machine learning. 
Additionally, the ANOVA evidence informed the choice of k, which helps to ensure that the selected features are the
most relevant for the model.

[Step 4: Duration 3.26 seconds| Input tokens: 10,372 | Output tokens: 158]

,Fold,Status,Method,ANOVA_Inspected,Selected_k,Selected_Features,Accuracy,Precision,Recall,F1,AUC,Agent_Runtime,ML_Runtime,Total_Model_Fits,Model_Evaluations,Input_Tokens,Output_Tokens,Total_Tokens,API_Cost_USD
0,1,completed,anova_kbest,True,5,"[perimeter1, concave_points1, radius3, perimeter3, concave_points3]",0.9474,1.0000,0.8596,0.9245,0.9937,6.1713,0.0818,1,1,10389,169,10558,0.0002
1,2,completed,anova_kbest,True,10,"[radius1, perimeter1, area1, compactness1, concavity1, concave_points1, radius3, perimeter3, area3, concave_points3]",0.9145,0.9231,0.8421,0.8807,0.9771,4.6450,0.0822,1,1,10446,109,10555,0.0002
2,3,completed,anova_kbest,True,10,"[radius1, perimeter1, area1, concavity1, concave_points1, radius3, perimeter3, area3, concavity3, concave_points3]",0.9470,0.9138,0.9464,0.9298,0.9801,6.0120,0.0370,1,1,10372,158,10530,0.0002


,Level,Agent_Runtime,Input_Tokens,Output_Tokens,Total_Tokens,API_Cost_USD
0,2,16.828334,31207,436,31643,0.000646


Level 2 Usage Summary

Final metrics and standard deviation

In [208]:
successful = level2_results_df[
    level2_results_df["Status"] == "completed"
]

if len(successful) == skfolds.get_n_splits():

    metrics = ["Accuracy", "Precision", "Recall", "F1", "AUC"]

    level2_summary = pd.DataFrame([{
        "Level": 2,
        **{
            metric: successful[metric].mean()
            for metric in metrics
        },
        "F1_std": successful["F1"].std(ddof=1),
        "Recall_std": successful["Recall"].std(ddof=1),
        "Agent_Runtime": level2_results_df["Agent_Runtime"].sum(),
        "ML_Runtime": successful["ML_Runtime"].sum(),
        "Total_Runtime": total_runtime,
        "Total_Model_Fits": successful["Total_Model_Fits"].sum()
    }])

    display(level2_summary.round(4))

else:
    print(
        "Level 2 incomplete:",
        len(successful),
        "of",
        skfolds.get_n_splits(),
        "folds completed."
    )

,Level,Accuracy,Precision,Recall,F1,AUC,F1_std,Recall_std,Agent_Runtime,ML_Runtime,Total_Runtime,Total_Model_Fits
0,2,0.9363,0.9456,0.8827,0.9117,0.9836,0.0269,0.0559,16.8283,0.2011,17.2155,3


Level 2 agent on all training data

In [ ]:
# # Reset Level 2 context for final training
# level2_context.clear()

# level2_context.update({
#     "X": X_train,
#     "y": y_train,
#     "fold": "final",
#     "method": None,
#     "features": None,
#     "k": None,
#     "anova_inspected": False,
#     "completed": False
# })

# # Reset agent execution history
# final_level2_result = l2_agent.run(
#     LEVEL2_PROMPT,
#     reset=True
# )

# # Verify the agent selected a path
# if level2_context["method"] is None:
#     raise ValueError(
#         "Level 2 agent did not select a feature-selection path."
#     )

# print("Selected method:", level2_context["method"])
# print("Selected k:", level2_context["k"])

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are performing Level 2: Agent-Directed Feature Selection.                                                   │
│                                                                                                                 │
│ GOAL:                                                                                                           │
│ Choose a feature-selection strategy for breast-cancer classification.                                           │
│                                                                                                                 │
│ The classifier is fixed as Logistic Regression.                                                                 │
│ You cannot change its model or hyperparameters.                                                                 │
│                                                                                                                 │
│ WORKFLOW:                                                                                                       │
│ 1. Inspect the available training features.                                                                     │
│ 2. Optionally inspect ANOVA rankings and scores.                                                                │
│ 3. Choose exactly ONE feature-selection path:                                                                   │
│    A. Select feature names directly using your knowledge.                                                       │
│    B. Choose k for ANOVA SelectKBest.                                                                           │
│    C. Use GridSearchCV to automatically determine k                                                             │
│       for ANOVA SelectKBest.                                                                                    │
│ 4. Once a valid path is selected, it is locked.                                                                 │
│ 5. Finish by explaining your chosen method.                                                                     │
│                                                                                                                 │
│ RULES:                                                                                                          │
│ - You decide whether ANOVA or automated search is necessary.                                                    │
│ - If choosing Path B, use available ANOVA evidence to inform k.                                                 │
│ - Consider the computational cost of each method.                                                               │
│ - Use only valid feature names.                                                                                 │
│ - Pass manual features as a list of separate strings.                                                           │
│ - Never change paths after selection.                                                                           │
│ - Never access the held-out test set.                                                                           │
│ - Do not invent evaluation results.                                                                             │
│                                                                                                                 │
│ FINAL ANSWER:                                                                                                   │
│ Report the selected path, features or k, and why you chose it.                                                  │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.1-8B-Instru

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'inspect_data' with arguments: {}                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {'rows': 455, 'features': 30, 'missing_values': 0, 'class_distribution': {0: 285, 1: 170}, 
'feature_names': |'radius1', 'texture1', 'perimeter1', 'area1', 'smoothness1', 'compactness1', 'concavity1', 
'concave_points1', 'symmetry1', 'fractal_dimension1', 'radius2', 'texture2', 'perimeter2', 'area2', 'smoothness2', 
'compactness2', 'concavity2', 'concave_points2', 'symmetry2', 'fractal_dimension2', 'radius3', 'texture3', 
'perimeter3', 'area3', 'smoothness3', 'compactness3', 'concavity3', 'concave_points3', 'symmetry3', 
'fractal_dimension3']}

[Step 1: Duration 0.67 seconds| Input tokens: 1,937 | Output tokens: 20]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (9 outputs left out)

Observations: Path B: ANOVA SelectKBest with k=5. The chosen method is ANOVA SelectKBest because it allows us to 
select the most relevant features based on their ANOVA F-scores and p-values, which is suitable for our 
breast-cancer classification task.

Final answer: Path B: ANOVA SelectKBest with k=5. The chosen method is ANOVA SelectKBest because it allows us to 
select the most relevant features based on their ANOVA F-scores and p-values, which is suitable for our 
breast-cancer classification task.

[Step 4: Duration 2.90 seconds| Input tokens: 10,465 | Output tokens: 150]

Selected method: anova_kbest
Selected k: 5


Fit the final Logistic Regression model

In [ ]:
# final_level2_model, final_features, final_k, final_fits = (
#     fit_level2_configuration(
#         X_train,
#         y_train
#     )
# )

# print("Final classifier: Logistic Regression")
# print("Feature-selection method:", level2_context["method"])
# print("Number of selected features:", final_k)
# print("Selected features:", final_features)
# print("Model fits:", final_fits)

Final classifier: Logistic Regression
Feature-selection method: anova_kbest
Number of selected features: 5
Selected features: ['perimeter1', 'concave_points1', 'radius3', 'perimeter3', 'concave_points3']
Model fits: 1


Save the fitted model and its configuration

In [ ]:
# import joblib

# final_level2_config = {
#     "level": 2,
#     "model": "LR",
#     "method": level2_context["method"],
#     "selected_features": final_features,
#     "selected_k": final_k,
#     "anova_inspected": level2_context["anova_inspected"],
#     "training_fits": final_fits,
#     "cv_folds": skfolds.n_splits,
#     "cv_random_state": skfolds.random_state,
#     "training_samples": len(X_train)
# }

# joblib.dump(
#     final_level2_model,
#     "level2_final_model.joblib"
# )

# joblib.dump(
#     final_level2_config,
#     "level2_final_config.joblib"
# )

# print("Final Level 2 model and configuration saved.")

Final Level 2 model and configuration saved.


final test for level 2

In [ ]:
# final_model = joblib.load("level2_final_model.joblib")
# final_config = joblib.load("level2_final_config.joblib")

# # Path A needs manual column selection
# if final_config["method"] == "manual":
#     X_test_eval = X_test[final_config["selected_features"]]
# else:
#     X_test_eval = X_test

# # Predict on held-out test observations
# y_pred = final_model.predict(X_test_eval)

# test_f1 = f1_score(y_test, y_pred)

# print("Level 2 Test F1:", test_f1)

#### Level 3 Model Selection and Tuning

In [221]:
MAX_L3_EVALUATIONS = 7


level3_context = {}
level3_action_log = []
level3_experiment_log = []
level3_fold_results = []

def reset_level3_context(X, y, fold):
    level3_context.clear()
    level3_context.update({
        "X": X,
        "y": y,
        "fold": fold,
        "method": None,
        "features": None,
        "k": None,
        "anova_inspected": False,
        "evaluations": 0,
        "evaluation_attempts": 0,
        "history": []
    })

Inspection and selection tools

In [223]:
@tool
def level3_inspect_data() -> str:
    """Return the current permitted training data summary."""
    X = level3_context["X"]
    y = level3_context["y"]

    return json.dumps({
        "rows": X.shape[0],
        "features": X.shape[1],
        "missing_values": int(X.isna().sum().sum()),
        "class_distribution": y.value_counts().to_dict(),
        "feature_names": X.columns.tolist()
    })


@tool
def level3_analyze_anova() -> str:
    """Return ANOVA feature rankings before feature selection."""
    if level3_context["method"] is not None:
        return "ERROR: Feature-selection path already locked."

    X = level3_context["X"]
    y = level3_context["y"]

    scores, p_values = f_classif(X, y)

    ranking = pd.DataFrame({
        "feature": X.columns,
        "f_score": scores,
        "p_value": p_values
    }).sort_values("f_score", ascending=False)

    level3_context["anova_inspected"] = True

    level3_action_log.append({
    "fold": level3_context["fold"],
    "action": "analyze_anova"
    })

    return ranking.to_json(orient="records")


@tool
def level3_select_features(features: list[str]) -> str:
    """
    Select features using existing knowledge.

    Args:
        features: List of valid feature-name strings.
    """
    if level3_context["method"] is not None:
        return "ERROR: Feature-selection path already locked."

    if level3_context["anova_inspected"]:
        return "ERROR: Knowledge-only selection unavailable after ANOVA."

    X = level3_context["X"]

    if not features or len(features) != len(set(features)):
        return "ERROR: Empty or duplicate feature list."

    invalid = [f for f in features if f not in X.columns]
    if invalid:
        return f"ERROR: Invalid features: {invalid}"

    level3_context.update({
        "method": "manual",
        "features": features.copy(),
        "k": len(features)
    })

    level3_action_log.append({
        "fold": level3_context["fold"],
        "action": "select_features_manual",
        "features": features.copy(),
        "n_features": len(features)
    })

    return f"Path A locked: {features}"


@tool
def level3_select_kbest(k: int) -> str:
    """
    Choose ANOVA SelectKBest with a fixed k.

    Args:
        k: Number of features to retain.
    """
    if level3_context["method"] is not None:
        return "ERROR: Feature-selection path already locked."

    if not 1 <= k <= level3_context["X"].shape[1]:
        return "ERROR: Invalid k."

    level3_context.update({
        "method": "anova_kbest",
        "features": None,
        "k": k
    })

    level3_action_log.append({
        "fold": level3_context["fold"],
        "action": "select_kbest",
        "k": k
    })

    return f"Path B locked: k={k}."


@tool
def level3_search_best_k() -> str:
    """Choose ANOVA SelectKBest with automated k selection."""
    if level3_context["method"] is not None:
        return "ERROR: Feature-selection path already locked."

    level3_context.update({
        "method": "anova_gridsearch",
        "features": None,
        "k": None
    })

    level3_action_log.append({
        "fold": level3_context["fold"],
        "action": "search_best_k",
        "k": None
})

    return "Path C locked: GridSearchCV will determine k."

In [224]:
PARAM_GRID = {
    "LR": {
        "C": [0.1, 1.0, 10.0]
    },

    "SVC": {
        "C": [0.1, 1.0, 10.0],
        "gamma": ["scale", "auto"]
    },

    "RF": {
        "n_estimators": [100, 300, 500],
        "max_depth": [None, 5, 10],
        "min_samples_split": [2, 5]
    },

    "GB": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5]
    },

    "XGB": {
        "n_estimators": [50, 100, 200],
        "max_depth": [3, 6],
        "learning_rate": [0.05, 0.1]
    }
}

validates the agent's requested model and hyperparameters

In [225]:
def configure_level3_model(model_name, hyperparameters):

    if model_name not in ML_MODELS:
        raise ValueError("Classifier not permitted.")

    if not isinstance(hyperparameters, dict):
        raise ValueError("Hyperparameters must be a dictionary.")

    allowed = PARAM_GRID[model_name]

    for name, value in hyperparameters.items():
        if name not in allowed:
            raise ValueError(f"Parameter '{name}' not permitted.")

        if not any(
            type(value) is type(v) and value == v
            for v in allowed[name]
        ):
            raise ValueError(
                f"Value {value} not permitted for {name}."
            )

    configured = clone(ML_MODELS[model_name])

    classifier_step = configured.steps[-1][0]

    configured.set_params(**{
        f"{classifier_step}__{name}": value
        for name, value in hyperparameters.items()
    })

    return configured

build and evaluate selected configuration

In [226]:
def build_level3_candidate(configured_model, X, cv):

    method = level3_context["method"]

    if method == "manual":
        return clone(configured_model)

    pipeline = Pipeline([
        ("selector", SelectKBest(f_classif)),
        ("classifier", clone(configured_model))
    ])

    if method == "anova_kbest":
        pipeline.set_params(selector__k=level3_context["k"])
        return pipeline

    if method == "anova_gridsearch":
        return GridSearchCV(
            estimator=pipeline,
            param_grid={
                "selector__k": list(range(1, X.shape[1] + 1))
            },
            scoring="f1",
            cv=cv,
            refit=True,
            n_jobs=-1
        )

    raise ValueError("Feature-selection path not selected.")

In [227]:
def evaluate_level3_candidate(model_name, configured_model):

    X = level3_context["X"]
    y = level3_context["y"]

    candidate = build_level3_candidate(
        configured_model,
        X,
        skfolds
    )

    X_eval = (
        X[level3_context["features"]]
        if level3_context["method"] == "manual"
        else X
    )

    result = evaluate_model(
        model_name=model_name,
        model=candidate,
        x=X_eval,
        y=y,
        cv=skfolds
    )

    if level3_context["method"] == "anova_gridsearch":

        n_candidates = X.shape[1]
        n_folds = skfolds.get_n_splits()

        result["CV_fits"] = (
            n_candidates * n_folds + 1
        ) * n_folds

        result["Feature_candidates"] = n_candidates * n_folds

    else:
        result["Feature_candidates"] = 0

    return result

evaluates l3 selected configuration using its previously selected feature strategy

In [228]:

@tool
def level3_evaluate_configuration(
    model_name: str,
    hyperparameters: dict
) -> str:
    """
    Evaluate a permitted classifier configuration using CV.

    Args:
        model_name: One of LR, SVC, RF, GB, XGB.
        hyperparameters: Dictionary of permitted parameter values.
            Use {} to retain pipeline defaults.
    """

    if level3_context["method"] is None:
        return "ERROR: Select a feature-selection path first."

    level3_context["evaluation_attempts"] += 1  

    if level3_context["evaluations"] >= MAX_L3_EVALUATIONS:
        return "ERROR: Evaluation budget exhausted."

    try:
        configured_model = configure_level3_model(
            model_name,
            hyperparameters
        )

        result = evaluate_level3_candidate(
            model_name,
            configured_model
        )

    except Exception as e:
        return f"ERROR: {e}"

    level3_context["evaluations"] += 1

    record = {
        "fold": level3_context["fold"],
        "model": model_name,
        "hyperparameters": hyperparameters.copy(),
        "feature_method": level3_context["method"],
        "features": level3_context["features"],
        "k": level3_context["k"],
        "f1_mean": result["F1_mean"],
        "f1_std": result["F1_std"],
        "accuracy": result["Accuracy_mean"],
        "precision": result["Precision_mean"],
        "recall": result["Recall_mean"],
        "auc": result["AUC_mean"],
        "runtime": result["Runtime_seconds"],
        "cv_fits": result["CV_fits"],
        "feature_candidates": result["Feature_candidates"]
    }

    level3_action_log.append({
        "fold": level3_context["fold"],
        "action": "evaluate_configuration",
        "model": model_name,
        "hyperparameters": hyperparameters.copy()
    })
    
    level3_context["history"].append(record)
    level3_experiment_log.append(record)

    return json.dumps(record, default=str)


Level 3 agent and prompt

In [229]:
l3_agent = ToolCallingAgent(
    tools=[
        level3_inspect_data,
        level3_analyze_anova,
        level3_select_features,
        level3_select_kbest,
        level3_search_best_k,
        level3_evaluate_configuration
    ],
    model=model,
    max_steps=15
)

In [230]:
LEVEL3_PROMPT = """
You are performing Level 3 of a controlled ML experiment.

GOAL:
Identify a strong classifier using training data only.

FEATURE SELECTION:
Choose ONE path:
A. Manual feature selection.
B. ANOVA SelectKBest with your chosen k.
C. ANOVA SelectKBest with automated k search.

You may inspect ANOVA scores before choosing.
Once selected, the feature strategy is locked.

MODEL SELECTION:
Choose among LR, SVC, RF, GB, and XGB.
Evaluate permitted model and hyperparameter configurations.

Use only parameter names and values from PARAM_GRID.
Unspecified parameters retain their pipeline values.

Use mean CV F1 as the primary selection metric.

RULES:
- Never access the held-out test set.
- Never change feature selection after locking it.
- Do not exceed the evaluation budget.
- Do not invent results.

REPORT:
Selected classifier, hyperparameters, feature method,
mean F1, F1 standard deviation, and configurations evaluated.
"""

In [231]:
LEVEL3_PROMPT_WITH_GRID = (
    LEVEL3_PROMPT
    + f"\nYou may successfully evaluate at most {MAX_L3_EVALUATIONS} configurations."
    + "\n\nPERMITTED HYPERPARAMETER GRID:\n"
    + json.dumps(PARAM_GRID, indent=2)
)

Fit the best configuration

In [232]:
def fit_level3_configuration(record, X, y, cv):

    configured = configure_level3_model(
        record["model"],
        record["hyperparameters"]
    )

    method = record["feature_method"]

    if method == "manual":
        features = record["features"]
        fitted = configured.fit(X[features], y)
        return fitted, features, len(features), 1

    pipeline = Pipeline([
        ("selector", SelectKBest(f_classif)),
        ("classifier", configured)
    ])

    if method == "anova_kbest":
        pipeline.set_params(selector__k=record["k"])
        fitted = pipeline.fit(X, y)
        fits = 1

    elif method == "anova_gridsearch":
        search = GridSearchCV(
            pipeline,
            {"selector__k": list(range(1, X.shape[1] + 1))},
            scoring="f1",
            cv=cv,
            refit=True,
            n_jobs=-1
        )
        search.fit(X, y)
        fitted = search.best_estimator_
        fits = len(search.cv_results_["params"]) * search.n_splits_ + 1

    else:
        raise ValueError("Unknown feature-selection method.")

    features = X.columns[
        fitted.named_steps["selector"].get_support()
    ].tolist()

    return fitted, features, len(features), fits

Run the three outer CV folds

In [233]:

level3_fold_results = []
level3_experiment_log = []
level3_action_log = []

start_total = time.perf_counter()

for fold, (train_idx, val_idx) in enumerate(
    skfolds.split(X_train, y_train), start=1
):

    X_outer_train = X_train.iloc[train_idx]
    y_outer_train = y_train.iloc[train_idx]

    X_outer_val = X_train.iloc[val_idx]
    y_outer_val = y_train.iloc[val_idx]

    reset_level3_context(X_outer_train, y_outer_train, fold)

    start_agent = time.perf_counter()

    try:
        l3_agent.run(LEVEL3_PROMPT_WITH_GRID, reset=True)

        agent_runtime = time.perf_counter() - start_agent

        level3_usage = get_agent_usage(l3_agent)

        if not level3_context["history"]:
            raise ValueError("No successful configurations evaluated.")

        best_record = max(
            level3_context["history"],
            key=lambda r: r["f1_mean"]
        )

        start_fit = time.perf_counter()

        fitted_model, features, selected_k, final_fits = (
            fit_level3_configuration(
                best_record,
                X_outer_train,
                y_outer_train,
                skfolds
            )
        )

        X_eval = (
            X_outer_val[features]
            if best_record["feature_method"] == "manual"
            else X_outer_val
        )

        y_pred = fitted_model.predict(X_eval)

        if hasattr(fitted_model, "predict_proba"):
            pos_idx = list(fitted_model.classes_).index(1)
            y_score = fitted_model.predict_proba(
                X_eval
            )[:, pos_idx]
        else:
            y_score = fitted_model.decision_function(X_eval)
            if fitted_model.classes_[1] != 1:
                y_score = -y_score

        fit_runtime = time.perf_counter() - start_fit

        level3_fold_results.append({
            "Fold": fold,
            "Status": "completed",
            "Model": best_record["model"],
            "Hyperparameters": best_record["hyperparameters"],
            "Feature_Method": best_record["feature_method"],
            "Selected_k": selected_k,
            "Selected_Features": features,
            "Evaluation_Attempts": level3_context["evaluation_attempts"],
            "Accuracy": accuracy_score(y_outer_val, y_pred),
            "Precision": precision_score(
                y_outer_val, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_outer_val, y_pred, zero_division=0
            ),
            "F1": f1_score(
                y_outer_val, y_pred, zero_division=0
            ),
            "AUC": roc_auc_score(y_outer_val, y_score),
            "Configurations_Evaluated": len(level3_context["history"]),
            "Development_Fits": sum(
                r["cv_fits"] for r in level3_context["history"]
            ),
            "Final_Training_Fits": final_fits,
            "Total_Model_Fits": (
                sum(r["cv_fits"] for r in level3_context["history"])
                + final_fits
            ),
            "Agent_Runtime": agent_runtime,
            "Final_Fit_Runtime": fit_runtime,

            "Input_Tokens": level3_usage["Input_Tokens"],
            "Output_Tokens": level3_usage["Output_Tokens"],
            "Total_Tokens": level3_usage["Total_Tokens"],
            "API_Cost_USD": level3_usage["API_Cost_USD"]
        })

    except Exception as e:
        level3_fold_results.append({
            "Fold": fold,
            "Status": "failed",
            "Error": str(e),
            "Evaluation_Attempts": level3_context["evaluation_attempts"],
            "Agent_Runtime": time.perf_counter() - start_agent,
            "Configurations_Evaluated": len(
                level3_context["history"]
            ),
            "Development_Fits": sum(
                r["cv_fits"]
                for r in level3_context["history"]
            )
        })

total_runtime = time.perf_counter() - start_total

level3_results_df = pd.DataFrame(level3_fold_results)
display(level3_results_df.round(6))

level3_usage_summary = pd.DataFrame([{
    "Level": 3,
    "Agent_Runtime": level3_results_df["Agent_Runtime"].sum(),
    "Input_Tokens": level3_results_df["Input_Tokens"].sum(min_count=1),
    "Output_Tokens": level3_results_df["Output_Tokens"].sum(min_count=1),
    "Total_Tokens": level3_results_df["Total_Tokens"].sum(min_count=1),
    "API_Cost_USD": level3_results_df["API_Cost_USD"].sum(min_count=1)
}])

display(level3_usage_summary.round(6))

level3_results_df.to_csv(
    "level3_run_1_results.csv",
    index=False
)

level3_usage_summary.to_csv(
    "level3_run_1_usage.csv",
    index=False
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are performing Level 3 of a controlled ML experiment.                                                       │
│                                                                                                                 │
│ GOAL:                                                                                                           │
│ Identify a strong classifier using training data only.                                                          │
│                                                                                                                 │
│ FEATURE SELECTION:                                                                                              │
│ Choose ONE path:                                                                                                │
│ A. Manual feature selection.                                                                                    │
│ B. ANOVA SelectKBest with your chosen k.                                                                        │
│ C. ANOVA SelectKBest with automated k search.                                                                   │
│                                                                                                                 │
│ You may inspect ANOVA scores before choosing.                                                                   │
│ Once selected, the feature strategy is locked.                                                                  │
│                                                                                                                 │
│ MODEL SELECTION:                                                                                                │
│ Choose among LR, SVC, RF, GB, and XGB.                                                                          │
│ Evaluate permitted model and hyperparameter configurations.                                                     │
│                                                                                                                 │
│ Use only parameter names and values from PARAM_GRID.                                                            │
│ Unspecified parameters retain their pipeline values.                                                            │
│                                                                                                                 │
│ Use mean CV F1 as the primary selection metric.                                                                 │
│                                                                                                                 │
│ RULES:                                                                                                          │
│ - Never access the held-out test set.                                                                           │
│ - Never change feature selection after locking it.                                                              │
│ - Do not exceed the evaluation budget.                                                                          │
│ - Do not invent results.                                                                                        │
│                                                                                                                 │
│ REPORT:                                                                                                         │
│ Selected classifier, hyperparameters, feature method,                                                           │
│ mean F1, F1 standard deviation, and configurations evaluated.                                                   │
│                                                       

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'level3_inspect_data' with arguments: {}                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"rows": 303, "features": 30, "missing_values": 0, "class_distribution": {"0": 190, "1": 113}, 
"feature_names": |"radius1", "texture1", "perimeter1", "area1", "smoothness1", "compactness1", "concavity1", 
"concave_points1", "symmetry1", "fractal_dimension1", "radius2", "texture2", "perimeter2", "area2", "smoothness2", 
"compactness2", "concavity2", "concave_points2", "symmetry2", "fractal_dimension2", "radius3", "texture3", 
"perimeter3", "area3", "smoothness3", "compactness3", "concavity3", "concave_points3", "symmetry3", 
"fractal_dimension3"]}

[Step 1: Duration 1.68 seconds| Input tokens: 2,354 | Output tokens: 13]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (74 outputs left out)

Final answer: The selected classifier is RF with hyperparameters n_estimators=300, max_depth=5, and 
min_samples_split=5. The mean F1 score is 0.9098013676326929, and the F1 standard deviation is 
0.031302546450577035. The configurations evaluated are RF with n_estimators=300, max_depth=5, and 
min_samples_split=5, and RF with n_estimators=500, max_depth=10, and min_samples_split=5.

[Step 8: Duration 11.81 seconds| Input tokens: 30,344 | Output tokens: 567]

,Fold,Status,Model,Hyperparameters,Feature_Method,Selected_k,Selected_Features,Evaluation_Attempts,Accuracy,Precision,...,Configurations_Evaluated,Development_Fits,Final_Training_Fits,Total_Model_Fits,Agent_Runtime,Final_Fit_Runtime,Input_Tokens,Output_Tokens,Total_Tokens,API_Cost_USD
0,1,completed,RF,"{'n_estimators': 300, 'max_depth': 5, 'min_sam...",anova_kbest,10,"[radius1, perimeter1, area1, concavity1, conca...",1,0.967105,0.964286,...,1,3,1,4,20.981992,0.773325,19911,323,20234,0.000414
1,2,completed,RF,"{'n_estimators': 300, 'max_depth': 5, 'min_sam...",anova_gridsearch,29,"[radius1, texture1, perimeter1, area1, smoothn...",1,0.927632,0.925926,...,1,273,91,364,29.912232,6.932144,16262,271,16533,0.000339
2,3,completed,RF,"{'n_estimators': 500, 'max_depth': 10, 'min_sa...",anova_kbest,10,"[radius1, perimeter1, area1, concavity1, conca...",3,0.947020,0.900000,...,3,9,1,10,47.141533,1.381484,30344,567,30911,0.000635


,Level,Agent_Runtime,Input_Tokens,Output_Tokens,Total_Tokens,API_Cost_USD
0,3,98.035757,66517,1161,67678,0.001388


Summary and SD

In [234]:
successful = level3_results_df[
    level3_results_df["Status"] == "completed"
]

if len(successful) == skfolds.get_n_splits():

    metrics = ["Accuracy", "Precision", "Recall", "F1", "AUC"]

    level3_summary = pd.DataFrame([{
        "Level": 3,
        **{
            metric: successful[metric].mean()
            for metric in metrics
        },
        "F1_std": successful["F1"].std(ddof=1),
        "Total_Runtime": total_runtime,
        "Configurations_Evaluated":
            successful["Configurations_Evaluated"].sum(),
        "Development_Fits":
            successful["Development_Fits"].sum(),
        "Final_Training_Fits":
            successful["Final_Training_Fits"].sum(),
        "Total_Model_Fits":
            successful["Total_Model_Fits"].sum(),
        "Evaluation_Attempts":
            level3_results_df["Evaluation_Attempts"].sum()
    }])

    display(level3_summary.round(4))

else:
    print("Level 3 evaluation incomplete.")

,Level,Accuracy,Precision,Recall,F1,AUC,F1_std,Total_Runtime,Configurations_Evaluated,Development_Fits,Final_Training_Fits,Total_Model_Fits,Evaluation_Attempts
0,3,0.9473,0.9301,0.9296,0.9292,0.9834,0.0275,107.2721,5,285,93,378,5


Fit and save the final Level 3 model

In [ ]:
# reset_level3_context(X_train, y_train, fold="final")

# start_final_agent = time.perf_counter()

# l3_agent.run(LEVEL3_PROMPT_WITH_GRID, reset=True)

# final_level3_agent_runtime = (
#     time.perf_counter() - start_final_agent
# )

# final_level3_usage = get_agent_usage(l3_agent)

# if not level3_context["history"]:
#     raise ValueError("No valid final configuration evaluated.")

# final_best = max(
#     level3_context["history"],
#     key=lambda r: r["f1_mean"]
# )

# final_model, final_features, final_k, final_fits = (
#     fit_level3_configuration(
#         final_best,
#         X_train,
#         y_train,
#         skfolds
#     )
# )

# final_config = {
#     "level": 3,
#     "model": final_best["model"],
#     "hyperparameters": final_best["hyperparameters"],
#     "feature_method": final_best["feature_method"],
#     "selected_features": final_features,
#     "selected_k": final_k,
#     "development_f1": final_best["f1_mean"],
#     "development_f1_std": final_best["f1_std"],
#     "training_fits": final_fits
# }

# joblib.dump(final_model, "level3_final_model.joblib")
# joblib.dump(final_config, "level3_final_config.joblib")

# print(final_config)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are performing Level 3 of a controlled ML experiment.                                                       │
│                                                                                                                 │
│ GOAL:                                                                                                           │
│ Identify a strong classifier using training data only.                                                          │
│                                                                                                                 │
│ FEATURE SELECTION:                                                                                              │
│ Choose ONE path:                                                                                                │
│ A. Manual feature selection.                                                                                    │
│ B. ANOVA SelectKBest with your chosen k.                                                                        │
│ C. ANOVA SelectKBest with automated k search.                                                                   │
│                                                                                                                 │
│ You may inspect ANOVA scores before choosing.                                                                   │
│ Once selected, the feature strategy is locked.                                                                  │
│                                                                                                                 │
│ MODEL SELECTION:                                                                                                │
│ Choose among LR, SVC, RF, GB, and XGB.                                                                          │
│ Evaluate permitted model and hyperparameter configurations.                                                     │
│                                                                                                                 │
│ Use only parameter names and values from PARAM_GRID.                                                            │
│ Unspecified parameters retain their pipeline values.                                                            │
│                                                                                                                 │
│ Use mean CV F1 as the primary selection metric.                                                                 │
│                                                                                                                 │
│ RULES:                                                                                                          │
│ - Never access the held-out test set.                                                                           │
│ - Never change feature selection after locking it.                                                              │
│ - Do not exceed the evaluation budget.                                                                          │
│ - Do not invent results.                                                                                        │
│                                                                                                                 │
│ REPORT:                                                                                                         │
│ Selected classifier, hyperparameters, feature method,                                                           │
│ mean F1, F1 standard deviation, and configurations evaluated.                                                   │
│                                                       

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'level3_analyze_anova' with arguments: {}                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 
|{"feature":"concave_points3","f_score":733.7249325739,"p_value":8.778594292e-97},{"feature":"perimeter3","f_score"
:717.2464871041,"p_value":2.093046266e-95},{"feature":"radius3","f_score":692.8613952606,"p_value":2.483688451e-93}
,{"feature":"concave_points1","f_score":684.5268452011,"p_value":1.300874686e-92},{"feature":"perimeter1","f_score"
:548.4132358503,"p_value":4.687520895e-80},{"feature":"area3","f_score":522.1889466673,"p_value":1.93238663e-77},{"
feature":"radius1","f_score":511.2748478448,"p_value":2.485052799e-76},{"feature":"area1","f_score":444.8575181971,
"p_value":2.691158241e-69},{"feature":"concavity1","f_score":397.592081765,"p_value":5.781926029e-64},{"feature":"c
oncavity3","f_score":319.507776201,"p_value":1.820807577e-54},{"feature":"compactness1","f_score":263.5643259368,"p
_value":4.787997937e-47},{"feature":"compactness3","f_score":238.2037248336,"p_value":1.732412388e-43},{"feature":"
radius2","f_score":205.4332063694,"p_value":1.091010279e-38},{"feature":"perimeter2","f_score":193.1689311221,"p_va
lue":7.879571682e-37},{"feature":"area2","f_score":180.5612294332,"p_value":6.998373308e-35},{"feature":"texture3",
"f_score":126.1191128835,"p_value":5.502942982e-26},{"feature":"smoothness3","f_score":103.7232652718,"p_value":4.5
01784053e-22},{"feature":"symmetry3","f_score":100.5593965319,"p_value":1.657176003e-21},{"feature":"texture1","f_s
core":93.4814798971,"p_value":3.14753898e-20},{"feature":"concave_points2","f_score":87.5060890764,"p_value":3.9006
1212e-19},{"feature":"smoothness1","f_score":80.5672911479,"p_value":7.530863648e-18},{"feature":"symmetry1","f_sco
re":59.7284676726,"p_value":0.0},{"feature":"fractal_dimension3","f_score":53.9055858621,"p_value":0.0},{"feature":
"compactness2","f_score":40.03288415,"p_value":0.0000000006},{"feature":"concavity2","f_score":24.7207527629,"p_val
ue":0.0000009419},{"feature":"fractal_dimension2","f_score":2.2526472187,"p_value":0.134082703},{"feature":"smoothn
ess2","f_score":1.4586834447,"p_value":0.2277698848},{"feature":"texture2","f_score":0.0201119137,"p_value":0.88728
80147},{"feature":"symmetry2","f_score":0.0187328469,"p_value":0.8911959475},{"feature":"fractal_dimension1","f_sco
re":0.0138741016,"p_value":0.9062875474}]

[Step 1: Duration 1.50 seconds| Input tokens: 2,354 | Output tokens: 37]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (9 outputs left out)

Observations: RF with hyperparameters: n_estimators=300, max_depth=10, min_samples_split=5, and feature method: 
ANOVA SelectKBest with k=5

Final answer: RF with hyperparameters: n_estimators=300, max_depth=10, min_samples_split=5, and feature method: 
ANOVA SelectKBest with k=5

[Step 4: Duration 2.56 seconds| Input tokens: 12,576 | Output tokens: 177]

{'level': 3, 'model': 'RF', 'hyperparameters': {'n_estimators': 300, 'max_depth': 10, 'min_samples_split': 5}, 'feature_method': 'anova_kbest', 'selected_features': ['perimeter1', 'concave_points1', 'radius3', 'perimeter3', 'concave_points3'], 'selected_k': 5, 'development_f1': 0.9171597633136095, 'development_f1_std': np.float64(0.02880331512193418), 'training_fits': 1}


#### Level 4 - Adaptive Workflow

imports and initializaiton

In [235]:
import json
import time
import joblib
import numpy as np
import pandas as pd

from copy import deepcopy
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
from smolagents import tool, ToolCallingAgent

In [236]:
LEVEL4_MODELS = {
    "LR": clone(LR_pipeline.steps[-1][1]),
    "SVC": clone(SVC_pipeline.steps[-1][1]),
    "RF": clone(RF_pipeline.steps[-1][1]),
    "GB": clone(GB_pipeline.steps[-1][1]),
    "XGB": clone(XGB_pipeline.steps[-1][1])
}

MAX_L4_EVALUATIONS = 5

level4_context = {}
level4_action_log = []
level4_experiment_log = []
level4_fold_results = []

Experiment state

In [ ]:
def reset_level4_context(X, y, fold):

    level4_context.clear()

    level4_context.update({
        "X": X,
        "y": y,
        "fold": fold,
        "evaluations": 0,
        "evaluation_attempts": 0,
        "history": []
    })

data inspection tool

In [237]:
@tool
def level4_inspect_data() -> str:
    """
    Inspect the current permitted training dataset.
    """

    X = level4_context["X"]
    y = level4_context["y"]

    summary = {
        "rows": X.shape[0],
        "features": X.shape[1],
        "numeric_features": X.select_dtypes(
            include=np.number
        ).columns.tolist(),
        "categorical_features": X.select_dtypes(
            exclude=np.number
        ).columns.tolist(),
        "missing_values": X.isna().sum().to_dict(),
        "class_distribution": y.value_counts().to_dict()
    }

    level4_action_log.append({
        "fold": level4_context["fold"],
        "action": "inspect_data"
    })

    return json.dumps(summary, default=str)

shared preprocessing 

In [238]:
def build_level4_preprocessor(X, imputation, encoding, scaling):

    if imputation not in [
        "none", "mean", "median", "most_frequent"
    ]:
        raise ValueError("Invalid imputation strategy.")

    if encoding not in ["none", "onehot"]:
        raise ValueError("Invalid encoding strategy.")

    if scaling not in ["none", "standard"]:
        raise ValueError("Invalid scaling strategy.")

    numeric_cols = X.select_dtypes(
        include=np.number
    ).columns.tolist()

    categorical_cols = X.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    if not numeric_cols and not categorical_cols:
        raise ValueError("No features available.")

    if categorical_cols and encoding != "onehot":
        raise ValueError(
            "Categorical features require one-hot encoding."
        )

    if imputation == "none" and X.isna().any().any():
        raise ValueError(
            "Missing values detected. Choose imputation."
        )

    transformers = []

    if numeric_cols:
        steps = []

        if imputation != "none":
            steps.append((
                "imputer",
                SimpleImputer(strategy=imputation)
            ))

        if scaling == "standard":
            steps.append((
                "scaler",
                StandardScaler()
            ))

        numeric_pipeline = (
            Pipeline(steps) if steps else "passthrough"
        )

        transformers.append((
            "numeric",
            numeric_pipeline,
            numeric_cols
        ))

    if categorical_cols:
        steps = []

        if imputation != "none":
            steps.append((
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ))

        steps.append((
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        ))

        transformers.append((
            "categorical",
            Pipeline(steps),
            categorical_cols
        ))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0
    )

Preprocessing tracking function

In [239]:
def track_level4_preprocessing(config, X):

    if config["feature_method"] == "manual":
        X = X[config["features"]]

    numeric = X.select_dtypes(include=np.number)
    categorical = X.select_dtypes(exclude=np.number)

    report = {}

    # IMPUTATION
    missing = int(X.isna().sum().sum())

    if config["imputation"] == "none":
        report["SimpleImputer"] = {
            "status": "Not requested"
        }
    elif missing == 0:
        report["SimpleImputer"] = {
            "status": "Requested but no changes applied",
            "reason": "No missing values",
            "values_changed": 0
        }
    else:
        report["SimpleImputer"] = {
            "status": "Applied",
            "strategy": config["imputation"],
            "values_changed": missing
        }

    # ONE-HOT ENCODING
    if config["encoding"] == "none":
        report["OneHotEncoder"] = {
            "status": "Not requested"
        }
    elif categorical.shape[1] == 0:
        report["OneHotEncoder"] = {
            "status": "Requested but no changes applied",
            "reason": "No categorical features",
            "values_changed": 0
        }
    else:
        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
        cat_data = categorical.copy()

        if config["imputation"] != "none":
            cat_data = pd.DataFrame(
                SimpleImputer(
                    strategy="most_frequent"
                ).fit_transform(cat_data),
                columns=cat_data.columns,
                index=cat_data.index
            )

        encoded = encoder.fit_transform(cat_data)

        report["OneHotEncoder"] = {
            "status": "Applied",
            "original_columns": categorical.shape[1],
            "encoded_columns": encoded.shape[1],
            "encoded_feature_names":
                encoder.get_feature_names_out().tolist()
        }

    # STANDARD SCALING
    if config["scaling"] == "none":
        report["StandardScaler"] = {
            "status": "Not requested"
        }
    elif numeric.empty:
        report["StandardScaler"] = {
            "status": "Requested but no changes applied",
            "reason": "No numerical features",
            "values_changed": 0
        }
    else:
        numeric_data = numeric.copy()

        if config["imputation"] != "none":
            numeric_data = pd.DataFrame(
                SimpleImputer(
                    strategy=config["imputation"]
                ).fit_transform(numeric_data),
                columns=numeric_data.columns,
                index=numeric_data.index
            )

        scaler = StandardScaler()
        scaled = scaler.fit_transform(numeric_data)

        before = numeric_data.to_numpy(dtype=float)

        changed = ~np.isclose(
            before, scaled,
            rtol=1e-7,
            atol=1e-9,
            equal_nan=True
        )

        report["StandardScaler"] = {
            "status": "Applied",
            "features_scaled": len(numeric.columns),
            "values_changed": int(changed.sum()),
            "example": {
                "feature": numeric.columns[0],
                "before": float(before[0, 0]),
                "after": float(scaled[0, 0])
            }
        }

    return report

optional ANOVA anlysis

In [240]:
@tool
def level4_analyze_anova(
    imputation: str = "none",
    encoding: str = "none",
    scaling: str = "none"
) -> str:
    """
    Inspect ANOVA rankings using optional preprocessing.

    Args:
        imputation: none, mean, median, or most_frequent.
        encoding: none or onehot.
        scaling: none or standard.
    """

    X = level4_context["X"]
    y = level4_context["y"]

    try:
        preprocessor = build_level4_preprocessor(
            X, imputation, encoding, scaling
        )

        X_processed = preprocessor.fit_transform(X)

        scores, p_values = f_classif(X_processed, y)

        ranking = pd.DataFrame({
            "feature": preprocessor.get_feature_names_out(),
            "f_score": scores,
            "p_value": p_values
        }).sort_values("f_score", ascending=False)

    except Exception as e:
        return f"ERROR: {e}"

    level4_action_log.append({
        "fold": level4_context["fold"],
        "action": "analyze_anova",
        "imputation": imputation,
        "encoding": encoding,
        "scaling": scaling
    })

    return ranking.to_json(orient="records")

Model and Hyperparameter validation

In [241]:
def configure_level4_model(model_name, hyperparameters):

    if model_name not in LEVEL4_MODELS:
        raise ValueError("Classifier not permitted.")

    if not isinstance(hyperparameters, dict):
        raise ValueError("Hyperparameters must be a dictionary.")

    allowed = PARAM_GRID[model_name]

    for name, value in hyperparameters.items():

        if name not in allowed:
            raise ValueError(f"Invalid parameter: {name}")

        if not any(
            type(value) is type(v) and value == v
            for v in allowed[name]
        ):
            raise ValueError(
                f"Invalid value for {name}: {value}"
            )

    configured = clone(LEVEL4_MODELS[model_name])
    configured.set_params(**hyperparameters)

    return configured

Dynamic pipeline builder

In [ ]:
def build_level4_candidate(config, X, cv):

    required = {
        "model_name", "imputation", "encoding",
        "scaling", "feature_method", "features",
        "k", "hyperparameters"
    }

    if not isinstance(config, dict) or set(config) != required:
        raise ValueError(
            f"Configuration must contain exactly {sorted(required)}"
        )

    method = config["feature_method"]

    if method not in [
        "manual", "anova_kbest", "anova_gridsearch"
    ]:
        raise ValueError("Invalid feature-selection method.")

    if method == "manual":

        features = config["features"]

        if (
            not isinstance(features, list)
            or not features
            or len(features) != len(set(features))
            or any(f not in X.columns for f in features)
        ):
            raise ValueError("Invalid manual features.")

        if config["k"] is not None:
            raise ValueError("Use k=null for manual selection.")

        X_schema = X[features]

    else:
        if config["features"] != []:
            raise ValueError(
                "Use an empty feature list for ANOVA paths."
            )

        X_schema = X

    if method == "anova_kbest":
        k = config["k"]

        if type(k) is not int or not 1 <= k <= X_schema.shape[1]:
            raise ValueError("Invalid k.")

    if method == "anova_gridsearch" and config["k"] is not None:
        raise ValueError("Use k=null for automated selection.")

    preprocessor = build_level4_preprocessor(
        X_schema,
        config["imputation"],
        config["encoding"],
        config["scaling"]
    )

    classifier = configure_level4_model(
        config["model_name"],
        config["hyperparameters"]
    )

    steps = [("preprocessor", preprocessor)]

    if method != "manual":
        steps.append((
            "selector",
            SelectKBest(score_func=f_classif)
        ))

    steps.append(("classifier", classifier))

    pipeline = Pipeline(steps)

    if method == "anova_kbest":
        pipeline.set_params(selector__k=config["k"])

    if method == "anova_gridsearch":

        # For the current numerical breast-cancer dataset,
        # preprocessing retains one column per input feature.
        # Reject categorical expansion rather than silently using an incorrect k range.
        if len(X_schema.select_dtypes(exclude=np.number).columns):
            raise ValueError(
                "Automated k search with categorical encoding "
                "requires a transformed-feature-count search."
            )

        return GridSearchCV(
            estimator=pipeline,
            param_grid={
                "selector__k": list(
                    range(1, X_schema.shape[1] + 1)
                )
            },
            scoring="f1",
            cv=cv,
            refit=True,
            n_jobs=-1,
            error_score="raise"
        )

    return pipeline

Development evaluation helper

In [243]:
def evaluate_level4_candidate(config, X, y, cv):

    candidate = build_level4_candidate(config, X, cv)

    X_eval = (
        X[config["features"]]
        if config["feature_method"] == "manual"
        else X
    )

    y_array = np.asarray(y)
    fold_results = []

    start = time.perf_counter()
    fit_count = 0
    feature_candidates = 0

    for train_idx, val_idx in cv.split(X_eval, y_array):

        fitted = clone(candidate)

        X_tr = X_eval.iloc[train_idx]
        X_val = X_eval.iloc[val_idx]

        y_tr = y_array[train_idx]
        y_val = y_array[val_idx]

        fitted.fit(X_tr, y_tr)

        if config["feature_method"] == "anova_gridsearch":
            n_candidates = len(fitted.cv_results_["params"])
            fit_count += (
                n_candidates * fitted.n_splits_ + 1
            )
            feature_candidates += n_candidates
        else:
            fit_count += 1

        y_pred = fitted.predict(X_val)

        if hasattr(fitted, "predict_proba"):
            pos_idx = list(fitted.classes_).index(1)
            y_score = fitted.predict_proba(
                X_val
            )[:, pos_idx]

        elif hasattr(fitted, "decision_function"):
            y_score = fitted.decision_function(X_val)

            if list(fitted.classes_)[1] != 1:
                y_score = -y_score

        else:
            raise ValueError("AUC scores unavailable.")

        fold_results.append({
            "Accuracy": accuracy_score(y_val, y_pred),
            "Precision": precision_score(
                y_val, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_val, y_pred, zero_division=0
            ),
            "F1": f1_score(
                y_val, y_pred, zero_division=0
            ),
            "AUC": roc_auc_score(y_val, y_score)
        })

    runtime = time.perf_counter() - start
    df = pd.DataFrame(fold_results)

    return {
        **{
            metric: df[metric].mean()
            for metric in [
                "Accuracy", "Precision", "Recall", "F1", "AUC"
            ]
        },
        "F1_std": df["F1"].std(ddof=1),
        "Runtime_seconds": runtime,
        "CV_fits": fit_count,
        "Feature_candidates": feature_candidates,
        "Fold_results": fold_results
    }

Adaptive evaluation tool

In [244]:
@tool
def level4_evaluate_configuration(config: dict) -> str:
    """
    Evaluate a complete ML configuration using development CV.

    Args:
        config: Dictionary with model_name, imputation,
            encoding, scaling, feature_method, features,
            k, and hyperparameters.
            model_name: LR, SVC, RF, GB, or XGB.
            imputation: none, mean, median, most_frequent.
            encoding: none or onehot.
            scaling: none or standard.
            feature_method: manual, anova_kbest, anova_gridsearch.
            features: List of raw feature names for manual
                selection; otherwise [].
            k: Integer for anova_kbest; otherwise null.
            hyperparameters: Dictionary of values from PARAM_GRID.
    """

    state = level4_context

    state["evaluation_attempts"] += 1

    if state["evaluations"] >= MAX_L4_EVALUATIONS:
        return "ERROR: Evaluation budget exhausted."

    try:
        result = evaluate_level4_candidate(
            config,
            state["X"],
            state["y"],
            skfolds
        )

    except Exception as e:
        level4_action_log.append({
            "fold": state["fold"],
            "action": "evaluation_failed",
            "configuration": deepcopy(config),
            "error": str(e)
        })
        return f"ERROR: {e}"

    state["evaluations"] += 1

    preprocessing_log = track_level4_preprocessing(
        config,
        state["X"]
    )

    record = {
        "fold": state["fold"],
        "evaluation": state["evaluations"],
        "configuration": deepcopy(config),
        "f1_mean": result["F1"],
        "f1_std": result["F1_std"],
        "accuracy": result["Accuracy"],
        "precision": result["Precision"],
        "recall": result["Recall"],
        "auc": result["AUC"],
        "runtime": result["Runtime_seconds"],
        "cv_fits": result["CV_fits"],
        "feature_candidates": result["Feature_candidates"],
        "Preprocessing_Changes": preprocessing_log,
    }

    state["history"].append(record)
    level4_experiment_log.append(record)

    level4_action_log.append({
        "fold": state["fold"],
        "action": "evaluate_configuration",
        "evaluation": state["evaluations"],
        "configuration": deepcopy(config)
    })

    return json.dumps(record, default=str)

agent and prompt

In [245]:
l4_agent = ToolCallingAgent(
    tools=[
        level4_inspect_data,
        level4_analyze_anova,
        level4_evaluate_configuration
    ],
    model=model,
    max_steps=20
)

In [246]:
LEVEL4_PROMPT = """
You are performing Level 4: Fully Adaptive ML Workflow.

GOAL:
Develop and improve a breast-cancer classification workflow
using only the permitted training data.

AVAILABLE CAPABILITIES:
- Inspect the dataset.
- Examine ANOVA feature rankings, optionally using
  preprocessing before ANOVA.
- Choose among LR, SVC, RF, GB, and XGB.
- Choose imputation, encoding, scaling, or none.
- Select features manually, using ANOVA SelectKBest,
  or using automated k search with GridSearchCV.
- Choose hyperparameters from PARAM_GRID if needed.
- Evaluate complete configurations using development CV.
- Revise previous decisions based on evaluation feedback if needed.

AUTONOMY:
You decide which tools to call and in what order.
You do not need to follow a fixed sequence.
You may skip unnecessary operations.
You may revise preprocessing, feature selection,
classifier, and hyperparameters between evaluations.

For each evaluation, submit a complete configuration:
{
  "model_name": "SVC",
  "imputation": "none",
  "encoding": "none",
  "scaling": "standard",
  "feature_method": "anova_kbest",
  "features": [],
  "k": 15,
  "hyperparameters": {"C": 1.0, "gamma": "scale"}
}

Use [] for features when not selecting manually.
Use null for k when not using fixed-k ANOVA.

CONSTRAINTS:
- Only the five permitted classifiers.
- Only hyperparameter values in PARAM_GRID.
- Do not change the existing data split or CV procedure.
- Do not access outer validation or held-out test data.
- Do not invent results.
- Use mean development F1 as the primary metric.
- Consider F1 standard deviation and computational cost.
- Preprocessing and feature selection must be fitted
  inside the training folds during evaluation.

You may stop early if additional evaluations are unnecessary.

FINAL ANSWER:
Report your best evaluated configuration, the changes
you made, mean F1, F1 standard deviation, and
the number of evaluations performed.
"""

LEVEL4_PROMPT_WITH_GRID = (
    LEVEL4_PROMPT
    + f"\nMaximum successful evaluations: {MAX_L4_EVALUATIONS}"
    + "\n\nPERMITTED HYPERPARAMETER GRID:\n"
    + json.dumps(PARAM_GRID, indent=2)
)

Fit the best configuration

In [247]:
def fit_level4_configuration(record, X, y, cv):

    config = record["configuration"]

    candidate = build_level4_candidate(
        config, X, cv
    )

    X_fit = (
        X[config["features"]]
        if config["feature_method"] == "manual"
        else X
    )

    candidate.fit(X_fit, y)

    if config["feature_method"] == "anova_gridsearch":
        fitted_pipeline = candidate.best_estimator_
        fits = (
            len(candidate.cv_results_["params"])
            * candidate.n_splits_ + 1
        )
    else:
        fitted_pipeline = candidate
        fits = 1

    if config["feature_method"] == "manual":
        selected_features = config["features"].copy()

    else:
        preprocessor = fitted_pipeline.named_steps[
            "preprocessor"
        ]

        transformed_names = preprocessor.get_feature_names_out()

        selector = fitted_pipeline.named_steps["selector"]

        selected_features = transformed_names[
            selector.get_support()
        ].tolist()

    return (
        candidate,
        selected_features,
        len(selected_features),
        fits
    )

helper to evaluate the fitted pipeline on the outer validation fold, which the agent has not seen.

In [248]:
def evaluate_level4_fitted_model(
    fitted_model, record, X_val, y_val
):

    config = record["configuration"]

    X_eval = (
        X_val[config["features"]]
        if config["feature_method"] == "manual"
        else X_val
    )

    y_pred = fitted_model.predict(X_eval)

    if hasattr(fitted_model, "predict_proba"):
        pos_idx = list(fitted_model.classes_).index(1)
        y_score = fitted_model.predict_proba(
            X_eval
        )[:, pos_idx]

    elif hasattr(fitted_model, "decision_function"):
        y_score = fitted_model.decision_function(X_eval)

        if list(fitted_model.classes_)[1] != 1:
            y_score = -y_score

    else:
        raise ValueError("AUC scores unavailable.")

    return {
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(
            y_val, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_val, y_pred, zero_division=0
        ),
        "AUC": roc_auc_score(y_val, y_score)
    }

Outer cross validation

In [251]:

level4_fold_results = []
level4_experiment_log = []
level4_action_log = []

start_total = time.perf_counter()

for fold, (train_idx, val_idx) in enumerate(
    skfolds.split(X_train, y_train), start=1
):

    X_outer_train = X_train.iloc[train_idx]
    y_outer_train = y_train.iloc[train_idx]

    X_outer_val = X_train.iloc[val_idx]
    y_outer_val = y_train.iloc[val_idx]

    reset_level4_context(
        X_outer_train,
        y_outer_train,
        fold
    )

    start_agent = time.perf_counter()

    try:
        l4_agent.run(
            LEVEL4_PROMPT_WITH_GRID,
            reset=True
        )

        agent_runtime = time.perf_counter() - start_agent

        level4_usage = get_agent_usage(l4_agent)

        history = level4_context["history"]

        if not history:
            raise ValueError(
                "No successful configurations evaluated."
            )

        best_record = max(
            history,
            key=lambda r: r["f1_mean"]
        )

        start_fit = time.perf_counter()

        fitted_model, features, selected_k, final_fits = (
            fit_level4_configuration(
                best_record,
                X_outer_train,
                y_outer_train,
                skfolds
            )
        )

        metrics = evaluate_level4_fitted_model(
            fitted_model,
            best_record,
            X_outer_val,
            y_outer_val
        )

        fit_runtime = time.perf_counter() - start_fit

        config = best_record["configuration"]

        preprocessing_log = track_level4_preprocessing(
            config,
            X_outer_train
        )

        development_fits = sum(
            r["cv_fits"] for r in history
        )

        level4_fold_results.append({
            "Fold": fold,
            "Status": "completed",
            "Model": config["model_name"],
            "Imputation": config["imputation"],
            "Encoding": config["encoding"],
            "Scaling": config["scaling"],
            "Preprocessing_Changes": preprocessing_log,
            "Feature_Method": config["feature_method"],
            "Hyperparameters": config["hyperparameters"],
            "Selected_k": selected_k,
            "Selected_Features": features,
            **metrics,
            "Evaluation_Attempts": level4_context[
                "evaluation_attempts"
            ],
            "Configurations_Evaluated": len(history),
            "Development_Fits": development_fits,
            "Final_Training_Fits": final_fits,
            "Total_Model_Fits": development_fits + final_fits,
            "Agent_Runtime": agent_runtime,
            "Final_Fit_Runtime": fit_runtime,

            "Input_Tokens": level4_usage["Input_Tokens"],
            "Output_Tokens": level4_usage["Output_Tokens"],
            "Total_Tokens": level4_usage["Total_Tokens"],
            "API_Cost_USD": level4_usage["API_Cost_USD"],
        })

    except Exception as e:
        get_agent_usage(l4_agent)

        level4_fold_results.append({
            "Fold": fold,
            "Status": "failed",
            "Error": str(e),
            "Evaluation_Attempts": level4_context[
                "evaluation_attempts"
            ],
            "Configurations_Evaluated": len(
                level4_context["history"]
            ),
            "Development_Fits": sum(
                r["cv_fits"]
                for r in level4_context["history"]
            ),
            "Agent_Runtime": time.perf_counter() - start_agent
        })

total_runtime = time.perf_counter() - start_total

level4_results_df = pd.DataFrame(level4_fold_results)

display(level4_results_df.round(6))

leveL4_usage_summary = pd.DataFrame([{
    "Level": 3,
    "Agent_Runtime": level4_results_df["Agent_Runtime"].sum(),
    "Input_Tokens": level4_results_df["Input_Tokens"].sum(min_count=1),
    "Output_Tokens": level4_results_df["Output_Tokens"].sum(min_count=1),
    "Total_Tokens": level4_results_df["Total_Tokens"].sum(min_count=1),
    "API_Cost_USD": level4_results_df["API_Cost_USD"].sum(min_count=1)
}])

display(level3_usage_summary.round(6))


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are performing Level 4: Fully Adaptive ML Workflow.                                                         │
│                                                                                                                 │
│ GOAL:                                                                                                           │
│ Develop and improve a breast-cancer classification workflow                                                     │
│ using only the permitted training data.                                                                         │
│                                                                                                                 │
│ AVAILABLE CAPABILITIES:                                                                                         │
│ - Inspect the dataset.                                                                                          │
│ - Examine ANOVA feature rankings, optionally using                                                              │
│   preprocessing before ANOVA.                                                                                   │
│ - Choose among LR, SVC, RF, GB, and XGB.                                                                        │
│ - Choose imputation, encoding, scaling, or none.                                                                │
│ - Select features manually, using ANOVA SelectKBest,                                                            │
│   or using automated k search with GridSearchCV.                                                                │
│ - Choose hyperparameters from PARAM_GRID if needed.                                                             │
│ - Evaluate complete configurations using development CV.                                                        │
│ - Revise previous decisions based on evaluation feedback if needed.                                             │
│                                                                                                                 │
│ AUTONOMY:                                                                                                       │
│ You decide which tools to call and in what order.                                                               │
│ You do not need to follow a fixed sequence.                                                                     │
│ You may skip unnecessary operations.                                                                            │
│ You may revise preprocessing, feature selection,                                                                │
│ classifier, and hyperparameters between evaluations.                                                            │
│                                                                                                                 │
│ For each evaluation, submit a complete configuration:                                                           │
│ {                                                                                                               │
│   "model_name": "SVC",                                                                                          │
│   "imputation": "none",                                                                                         │
│   "encoding": "none",                                                                                           │
│   "scaling": "standard",                                                                                        │
│   "feature_method": "anova_kbest",                                                                              │
│   "features": [],                                     

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'level4_inspect_data' with arguments: {}                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"rows": 303, "features": 30, "numeric_features": |"radius1", "texture1", "perimeter1", "area1", 
"smoothness1", "compactness1", "concavity1", "concave_points1", "symmetry1", "fractal_dimension1", "radius2", 
"texture2", "perimeter2", "area2", "smoothness2", "compactness2", "concavity2", "concave_points2", "symmetry2", 
"fractal_dimension2", "radius3", "texture3", "perimeter3", "area3", "smoothness3", "compactness3", "concavity3", 
"concave_points3", "symmetry3", "fractal_dimension3"], "categorical_features": |], "missing_values": {"radius1": 0,
"texture1": 0, "perimeter1": 0, "area1": 0, "smoothness1": 0, "compactness1": 0, "concavity1": 0, 
"concave_points1": 0, "symmetry1": 0, "fractal_dimension1": 0, "radius2": 0, "texture2": 0, "perimeter2": 0, 
"area2": 0, "smoothness2": 0, "compactness2": 0, "concavity2": 0, "concave_points2": 0, "symmetry2": 0, 
"fractal_dimension2": 0, "radius3": 0, "texture3": 0, "perimeter3": 0, "area3": 0, "smoothness3": 0, 
"compactness3": 0, "concavity3": 0, "concave_points3": 0, "symmetry3": 0, "fractal_dimension3": 0}, 
"class_distribution": {"0": 190, "1": 113}}

[Step 1: Duration 1.26 seconds| Input tokens: 2,273 | Output tokens: 13]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (70 outputs left out)

Final answer: The best evaluated configuration is SVC with imputation as none, encoding as none, scaling as 
standard, feature method as anova_kbest, features as [], k as 10, and hyperparameters as {'C': 1.0, 'gamma': 
'scale'} with a mean F1 score of 0.8972105016408812, F1 standard deviation of 0.01690813192878709, accuracy of 
0.9243512586552772, precision of 0.9218077474892395, recall of 0.8771929824561404, AUC of 0.9843706488443331, and 
runtime of 0.23502209997968748 seconds.

[Step 6: Duration 7.37 seconds| Input tokens: 23,012 | Output tokens: 750]

,Fold,Status,Model,Imputation,Encoding,Scaling,Preprocessing_Changes,Feature_Method,Hyperparameters,Selected_k,...,Configurations_Evaluated,Development_Fits,Final_Training_Fits,Total_Model_Fits,Agent_Runtime,Final_Fit_Runtime,Input_Tokens,Output_Tokens,Total_Tokens,API_Cost_USD
0,1,completed,SVC,none,none,standard,"{'SimpleImputer': {'status': 'Not requested'},...",anova_kbest,"{'C': 1.0, 'gamma': 'scale'}",15,...,1,3,1,4,16.103189,0.147664,17409,405,17814,0.000368
1,2,completed,SVC,none,none,standard,"{'SimpleImputer': {'status': 'Not requested'},...",anova_kbest,"{'C': 1.0, 'gamma': 'scale'}",15,...,2,6,1,7,24.330188,0.079319,27554,605,28159,0.000581
2,3,completed,SVC,none,none,standard,"{'SimpleImputer': {'status': 'Not requested'},...",anova_kbest,"{'C': 1.0, 'gamma': 'scale'}",10,...,1,3,1,4,29.130815,0.058906,23012,750,23762,0.000498


,Level,Agent_Runtime,Input_Tokens,Output_Tokens,Total_Tokens,API_Cost_USD
0,3,98.035757,66517,1161,67678,0.001388


get summary of performance

In [ ]:
successful = level4_results_df[
    level4_results_df["Status"] == "completed"
]

if len(successful) == skfolds.get_n_splits():

    metrics = ["Accuracy", "Precision", "Recall", "F1", "AUC"]

    level4_summary = pd.DataFrame([{
        "Level": 4,
        **{
            metric: successful[metric].mean()
            for metric in metrics
        },
        "F1_std": successful["F1"].std(ddof=1),
        "Recall_std": successful["Recall"].std(ddof=1),
        "Total_Runtime": total_runtime,
        "Agent_Runtime": successful["Agent_Runtime"].sum(),
        "Evaluation_Attempts":
            successful["Evaluation_Attempts"].sum(),
        "Configurations_Evaluated":
            successful["Configurations_Evaluated"].sum(),
        "Development_Fits":
            successful["Development_Fits"].sum(),
        "Final_Training_Fits":
            successful["Final_Training_Fits"].sum(),
        "Total_Model_Fits":
            successful["Total_Model_Fits"].sum(),
        
    }])

    display(level4_summary.round(4))

    level4_usage_summary = pd.DataFrame([{
    "Level": 4,
    "Agent_Runtime": level4_results_df["Agent_Runtime"].sum(),
    "Input_Tokens": level4_results_df["Input_Tokens"].sum(min_count=1),
    "Output_Tokens": level4_results_df["Output_Tokens"].sum(min_count=1),
    "Total_Tokens": level4_results_df["Total_Tokens"].sum(min_count=1),
    "API_Cost_USD": level4_results_df["API_Cost_USD"].sum(min_count=1)
    }])

    display(level4_usage_summary.round(6))

else:
    print(
        "Level 4 incomplete:",
        len(successful),
        "of",
        skfolds.get_n_splits(),
        "folds completed."
    )

level4_results_df.to_csv(
    "level4_run_1_results.csv",
    index=False
)

level4_usage_summary.to_csv(
    "level4_run_1_usage.csv",
    index=False
)

,Level,Accuracy,Precision,Recall,F1,AUC,F1_std,Recall_std,Total_Runtime,Agent_Runtime,Evaluation_Attempts,Configurations_Evaluated,Development_Fits,Final_Training_Fits,Total_Model_Fits
0,4,0.9429,0.9392,0.906,0.9221,0.9866,0.0365,0.0402,70.6058,70.1627,8,4,12,3,15


Train on the 455 data and save the model 

In [ ]:
# reset_level4_context(
#     X_train,
#     y_train,
#     fold="final"
# )

# final_level4_result = l4_agent.run(
#     LEVEL4_PROMPT_WITH_GRID,
#     reset=True
# )

# if not level4_context["history"]:
#     raise ValueError("No valid final configuration evaluated.")

# final_best = max(
#     level4_context["history"],
#     key=lambda r: r["f1_mean"]
# )

# final_model, final_features, final_k, final_fits = (
#     fit_level4_configuration(
#         final_best,
#         X_train,
#         y_train,
#         skfolds
#     )
# )

# config = final_best["configuration"]

# final_preprocessing_log = track_level4_preprocessing(
#     config,
#     X_train
# )

# final_level4_config = {
#     "level": 4,
#     "model": config["model_name"],
#     "preprocessing": {
#         "imputation": config["imputation"],
#         "encoding": config["encoding"],
#         "scaling": config["scaling"]
#     },
#     "Preprocessing_Changes": final_preprocessing_log,
#     "feature_method": config["feature_method"],
#     "manual_features": config["features"],
#     "selected_features": final_features,
#     "selected_k": final_k,
#     "hyperparameters": config["hyperparameters"],
#     "development_f1": final_best["f1_mean"],
#     "development_f1_std": final_best["f1_std"],
#     "training_fits": final_fits,
#     "cv_folds": skfolds.n_splits,
#     "cv_random_state": skfolds.random_state,
#     "training_samples": len(X_train)
# }

# final_level4_config["full_configuration"] = config.copy()

# joblib.dump(
#     final_model,
#     "level4_final_model.joblib"
# )

# joblib.dump(
#     final_level4_config,
#     "level4_final_config.joblib"
# )

# print(final_level4_config)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are performing Level 4: Fully Adaptive ML Workflow.                                                         │
│                                                                                                                 │
│ GOAL:                                                                                                           │
│ Develop and improve a breast-cancer classification workflow                                                     │
│ using only the permitted training data.                                                                         │
│                                                                                                                 │
│ AVAILABLE CAPABILITIES:                                                                                         │
│ - Inspect the dataset.                                                                                          │
│ - Examine ANOVA feature rankings, optionally using                                                              │
│   preprocessing before ANOVA.                                                                                   │
│ - Choose among LR, SVC, RF, GB, and XGB.                                                                        │
│ - Choose imputation, encoding, scaling, or none.                                                                │
│ - Select features manually, using ANOVA SelectKBest,                                                            │
│   or using automated k search with GridSearchCV.                                                                │
│ - Choose hyperparameters from PARAM_GRID if needed.                                                             │
│ - Evaluate complete configurations using development CV.                                                        │
│ - Revise previous decisions based on evaluation feedback if needed.                                             │
│                                                                                                                 │
│ AUTONOMY:                                                                                                       │
│ You decide which tools to call and in what order.                                                               │
│ You do not need to follow a fixed sequence.                                                                     │
│ You may skip unnecessary operations.                                                                            │
│ You may revise preprocessing, feature selection,                                                                │
│ classifier, and hyperparameters between evaluations.                                                            │
│                                                                                                                 │
│ For each evaluation, submit a complete configuration:                                                           │
│ {                                                                                                               │
│   "model_name": "SVC",                                                                                          │
│   "imputation": "none",                                                                                         │
│   "encoding": "none",                                                                                           │
│   "scaling": "standard",                                                                                        │
│   "feature_method": "anova_kbest",                                                                              │
│   "features": [],                                     

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'level4_inspect_data' with arguments: {}                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"rows": 455, "features": 30, "numeric_features": |"radius1", "texture1", "perimeter1", "area1", 
"smoothness1", "compactness1", "concavity1", "concave_points1", "symmetry1", "fractal_dimension1", "radius2", 
"texture2", "perimeter2", "area2", "smoothness2", "compactness2", "concavity2", "concave_points2", "symmetry2", 
"fractal_dimension2", "radius3", "texture3", "perimeter3", "area3", "smoothness3", "compactness3", "concavity3", 
"concave_points3", "symmetry3", "fractal_dimension3"], "categorical_features": |], "missing_values": {"radius1": 0,
"texture1": 0, "perimeter1": 0, "area1": 0, "smoothness1": 0, "compactness1": 0, "concavity1": 0, 
"concave_points1": 0, "symmetry1": 0, "fractal_dimension1": 0, "radius2": 0, "texture2": 0, "perimeter2": 0, 
"area2": 0, "smoothness2": 0, "compactness2": 0, "concavity2": 0, "concave_points2": 0, "symmetry2": 0, 
"fractal_dimension2": 0, "radius3": 0, "texture3": 0, "perimeter3": 0, "area3": 0, "smoothness3": 0, 
"compactness3": 0, "concavity3": 0, "concave_points3": 0, "symmetry3": 0, "fractal_dimension3": 0}, 
"class_distribution": {"0": 285, "1": 170}}

[Step 1: Duration 0.63 seconds| Input tokens: 2,273 | Output tokens: 13]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (17 outputs left out)

Observations: The best evaluated configuration is SVC with imputation=none, encoding=none, scaling=standard, 
feature_method=anova_kbest, features=|], k=3, and hyperparameters={'C': 1.0, 'gamma': 'scale'}. The mean F1 score 
is 0.9240968240968241, the F1 standard deviation is 0.02898551177483237, and the number of evaluations performed is
2.

Final answer: The best evaluated configuration is SVC with imputation=none, encoding=none, scaling=standard, 
feature_method=anova_kbest, features=[], k=3, and hyperparameters={'C': 1.0, 'gamma': 'scale'}. The mean F1 score 
is 0.9240968240968241, the F1 standard deviation is 0.02898551177483237, and the number of evaluations performed is
2.

[Step 6: Duration 4.78 seconds| Input tokens: 22,529 | Output tokens: 509]

{'level': 4, 'model': 'SVC', 'preprocessing': {'imputation': 'none', 'encoding': 'none', 'scaling': 'standard'}, 'Preprocessing_Changes': {'SimpleImputer': {'status': 'Not requested'}, 'OneHotEncoder': {'status': 'Not requested'}, 'StandardScaler': {'status': 'Applied', 'features_scaled': 30, 'values_changed': 13650, 'example': {'feature': 'radius1', 'before': 16.02, 'after': 0.5185587269315232}}}, 'feature_method': 'anova_kbest', 'manual_features': [], 'selected_features': ['numeric__radius3', 'numeric__perimeter3', 'numeric__concave_points3'], 'selected_k': 3, 'hyperparameters': {'C': 1.0, 'gamma': 'scale'}, 'development_f1': np.float64(0.9240968240968241), 'development_f1_std': np.float64(0.02898551177483237), 'training_fits': 1, 'cv_folds': 3, 'cv_random_state': 42, 'training_samples': 455, 'full_configuration': {'model_name': 'SVC', 'imputation': 'none', 'encoding': 'none', 'scaling': 'standard', 'feature_method': 'anova_kbest', 'features': [], 'k': 3, 'hyperparameters': {'C': 1.0

In [ ]:
# saved_config = joblib.load("level4_final_config.joblib")

# print(saved_config["full_configuration"])